## 0) DOE 데이터 재생성 (선택)

새 DOE 데이터를 만들 때만 실행합니다.

- 실행 환경: **호스트 Motor-CAD** (Docker 아님)
- 생성 결과:
  - `case_NNNN/*.mot` : 케이스별 기준 모델 복사본
  - `case_NNNN/FEResultsData/` : Motor-CAD 원본 결과 루트
  - `case_NNNN/doe_condition.json` : DOE 변수/적용값
  - `case_NNNN/postproc/*.txt` : 메인 DOE 배치 산출물
  - `case_NNNN/postproc/*.h5` : 필요 시 0-D 후속 셀에서 생성

권장 순서:
1. 0-B 설정 셀에서 경로/샘플 수/단계를 지정
2. `RUN_DOE=True`로 변경
3. 0-C 실행 셀 실행 (`solve + txt`)
4. 학습까지 이어갈 때만 `RUN_H5_EXPORT=True`로 바꿔 0-D 실행

**중간 중단 후 복구:**
- 0-E-1 셀로 상태 점검 (`.mes` 있음 / `.txt` 없음 케이스 식별)
- 0-E-2 셀에서 `RUN_REPAIR=True`로 바꿔 `.txt` 재내보내기

In [ ]:

# ── Pipeline Overview: Preprocessing → H5 → Manifest → Docker Training ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import matplotlib.patheffects as pe
from pathlib import Path
import json

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

# ── read current state ─────────────────────────────────────────────────────
_doe = Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData")
_manifest_path = _doe / "doe_manifest.json"
_manifest = json.loads(_manifest_path.read_text(encoding="utf-8")) if _manifest_path.exists() else {}
_cases = _manifest.get("cases", [])
_n_disk   = len(list(_doe.glob("case_*")))
_n_mani   = len(_cases)
_n_h5     = sum(1 for c in _cases if c.get("h5_paths"))
_n_txt    = sum(1 for c in _cases if c.get("txt_paths"))

# ── figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 11))
ax.set_xlim(0, 22); ax.set_ylim(0, 11); ax.axis("off")
ax.set_facecolor("#f2f5fc"); fig.patch.set_facecolor("#f2f5fc")

C = dict(step="#2471a3", file="#1e8449", mani="#7d3c98",
         ckpt="#117a65", host="#212f3d", docker="#1a5276",
         bg_h="#dce8f5", bg_d="#d5eaf7")

def box(ax, x, y, w, h, fc, ec=None, alpha=0.93, r=0.3, lw=1.8, z=3):
    ax.add_patch(FancyBboxPatch((x,y), w, h,
        boxstyle=f"round,pad=0.05,rounding_size={r}",
        linewidth=lw, edgecolor=ec or fc,
        facecolor=fc, alpha=alpha, zorder=z))

def t(ax, x, y, s, sz=9, c="white", bold=False, ha="center", va="center", z=5):
    ax.text(x, y, s, fontsize=sz, color=c, ha=ha, va=va, zorder=z,
            fontweight="bold" if bold else "normal",
            path_effects=[pe.withStroke(linewidth=1.6, foreground="#00000018")])

def arr(ax, x0, y0, x1, y1, c="#444", lw=2.2, s="->", rad=0.0):
    ax.annotate("", xy=(x1,y1), xytext=(x0,y0),
        arrowprops=dict(arrowstyle=s, color=c, lw=lw,
                        connectionstyle=f"arc3,rad={rad}"), zorder=4)

# ── background regions ─────────────────────────────────────────────────────
box(ax, 0.3,0.3, 13.5,10.3, C["bg_h"], ec=C["host"], alpha=0.45, r=0.45, lw=2.5)
t(ax, 7.05,10.35, "Host  (Windows Workstation — D:/KDH/)", sz=12, c=C["host"], bold=True)

box(ax, 14.2,0.3, 7.3,10.3, C["bg_d"], ec=C["docker"], alpha=0.45, r=0.45, lw=2.5)
t(ax, 17.85,10.35, "Docker Container  (motor_compare)", sz=12, c=C["docker"], bold=True)

# ══ STEP 1 ── DOE Simulation ════════════════════════════════════════════════
box(ax, 0.6,7.9, 3.6,1.8, C["step"])
t(ax, 2.4,9.15, "STEP 1  DOE Simulation", sz=10.5, bold=True)
t(ax, 2.4,8.65, "doe_batch_run()", sz=9, c="#d6eaf8")
t(ax, 2.4,8.2,  f"Motor-CAD  N={_n_disk} cases", sz=8.5)

box(ax, 0.6,5.55, 1.65,2.1, C["file"])
t(ax, 1.42,6.95, "case_NNNN/", sz=8.5, bold=True)
t(ax, 1.42,6.6,  "*.mes  (FEA result)", sz=7.5)
t(ax, 1.42,6.3,  "Mag_*.txt  (export)", sz=7.5)
t(ax, 1.42,6.0,  "doe_condition.json", sz=7.5)
t(ax, 1.42,5.72, "(geometry/electrical)", sz=7, c="#ccffcc")

box(ax, 2.55,5.55, 1.65,2.1, C["mani"])
t(ax, 3.37,6.95, "doe_manifest.json", sz=8.5, bold=True)
t(ax, 3.37,6.6,  "(1st auto-created)", sz=8)
t(ax, 3.37,6.3,  "34 cases only", sz=8, c="#ffddff")
t(ax, 3.37,6.0,  "h5_paths: MISSING", sz=8, c="#ffaaaa")
t(ax, 3.37,5.72, "-> needs rebuild", sz=7, c="#ffcccc")

arr(ax, 2.4,7.9, 1.6,7.65, lw=1.8)
arr(ax, 2.4,7.9, 3.37,7.65, lw=1.8)

# ══ STEP 2 ── TXT → H5 Conversion ══════════════════════════════════════════
box(ax, 4.8,7.9, 4.0,1.8, C["step"])
t(ax, 6.8,9.15, "STEP 2  TXT -> H5 Conversion", sz=10.5, bold=True)
t(ax, 6.8,8.65, "doe_h5_batch_from_txt()", sz=9, c="#d6eaf8")
t(ax, 6.8,8.2,  "ProcessPoolExecutor  workers=8  ~ 48 sec", sz=8.5)

box(ax, 4.8,5.55, 3.85,2.1, C["file"])
t(ax, 6.72,6.95, "postproc/*.h5", sz=8.5, bold=True)
t(ax, 6.72,6.65, "Mag_OnLoadTorque_*.h5", sz=8)
t(ax, 6.72,6.38, "Mag_StaticLoad_*.h5", sz=8)
t(ax, 6.72,6.12, "Mag_StaticOC_*.h5", sz=8)
t(ax, 6.72,5.85, "/slideband/  CSR group  +", sz=7.5, c="#ccffcc")
t(ax, 6.72,5.62, "per-step re-mesh stored  v", sz=7, c="#ccffcc")

arr(ax, 4.2,8.8, 4.8,8.8, lw=2.2)
arr(ax, 6.8,7.9, 6.72,7.65, lw=1.8)
arr(ax, 5.7,5.55, 3.55,5.1, lw=1.5, c=C["mani"], s="-|>", rad=-0.1)
t(ax, 4.6,5.1, "h5_paths update", sz=7.5, c=C["mani"])

# ══ STEP 3 ── Manifest Rebuild ══════════════════════════════════════════════
box(ax, 9.3,7.9, 4.0,1.8, C["step"])
t(ax, 11.3,9.15, "STEP 3  Manifest Rebuild", sz=10.5, bold=True)
t(ax, 11.3,8.65, "_rebuild_manifest.py", sz=9, c="#d6eaf8")
t(ax, 11.3,8.2,  f"disk scan -> {_n_disk} cases  < 1 sec", sz=8.5)

box(ax, 9.3,5.55, 3.9,2.1, C["mani"])
t(ax, 11.25,6.95, "doe_manifest.json  (final)", sz=8.5, bold=True)
t(ax, 11.25,6.65, f"n_cases: {_n_mani}  (all cases)", sz=8)
t(ax, 11.25,6.38, f"h5_paths: {_n_h5} cases  v", sz=8)
t(ax, 11.25,6.12, f"txt_paths: {_n_txt} cases  v", sz=8)
t(ax, 11.25,5.85, "geometry + electrical  v", sz=8)
t(ax, 11.25,5.62, "case_0000~0039  all  v", sz=7.5, c="#e8d5ff")

arr(ax, 8.8,8.8, 9.3,8.8, lw=2.2)
arr(ax, 11.3,7.9, 11.25,7.65, lw=1.8)

# ── volume mount divider ────────────────────────────────────────────────────
ax.plot([14.05,14.05], [0.55,10.15], color="#888", lw=2.0, ls="--", zorder=2, alpha=0.55)
t(ax, 14.05,0.44, "Volume Mount  :ro", sz=8.5, c="#666")

arr(ax, 14.05,6.5, 14.5,6.5, lw=3.0, c=C["docker"], s="-|>")
t(ax, 14.27,6.82, "/workspace/doe_data", sz=8.5, c=C["docker"])

# ══ STEP 4 ── GNN Training (Docker) ════════════════════════════════════════
box(ax, 14.5,7.9, 6.7,1.8, C["docker"])
t(ax, 17.85,9.15, "STEP 4  GNN Training  (Docker)", sz=10.5, bold=True)
t(ax, 17.85,8.65, "train_meshgraphnet.py", sz=9, c="#cce8ff")
t(ax, 17.85,8.2,  "data_dir=/workspace/doe_data   GPU (RTX 3090)", sz=8.5)

box(ax, 14.5,5.55, 3.4,2.1, C["docker"])
t(ax, 16.2,6.95, "load_doe_data()", sz=8.5, bold=True)
t(ax, 16.2,6.65, "/workspace/doe_data", sz=8)
t(ax, 16.2,6.38, f"{_n_mani} cases  x  45 steps", sz=8)
t(ax, 16.2,6.12, f"~= {_n_mani*45:,} records", sz=9, c="#ffe066", bold=True)
t(ax, 16.2,5.85, "SB connectivity merged  v", sz=7.5, c="#ccffcc")
t(ax, 16.2,5.62, "per-step node positions  v", sz=7, c="#ccffcc")

box(ax, 18.2,5.55, 3.0,2.1, C["ckpt"])
t(ax, 19.7,6.95, "Output", sz=8.5, bold=True)
t(ax, 19.7,6.65, "doe_meshgraphnet_ckpt.pt", sz=7.5)
t(ax, 19.7,6.38, "/workspace/out/", sz=8)
t(ax, 19.7,6.12, "training log  (W&B)", sz=8)
t(ax, 19.7,5.85, "loss / metric plots", sz=8, c="#ccffcc")
t(ax, 19.7,5.62, "D:/KDH/EveryMotorOut/", sz=7, c="#aaffaa")

arr(ax, 14.05,8.8, 14.5,8.8, lw=2.5, c=C["docker"])
arr(ax, 17.85,7.9, 16.5,7.65, lw=1.8, c="white")
arr(ax, 17.85,7.9, 19.2,7.65, lw=1.8, c="white")

# manifest -> load_doe_data
ax.annotate("", xy=(14.9,5.55), xytext=(13.15,6.0),
    arrowprops=dict(arrowstyle="-|>", color=C["mani"], lw=1.8,
                    connectionstyle="arc3,rad=-0.25"), zorder=4)

# ── bottom summary table ────────────────────────────────────────────────────
box(ax, 0.6,0.5, 13.1,2.4, "#1a1f2e", alpha=0.88, lw=1.5)
t(ax, 1.0,2.65, "Execution Summary", sz=10, bold=True, c="#aaccff", ha="left")
rows = [
    ("STEP 1  DOE Sim",
     f"Motor-CAD COM  {_n_disk} cases x ~2min  =  serial ~80min / parallel ~10min  (PARALLEL_WORKERS=8)"),
    ("STEP 2  H5 Conv",
     "doe_h5_batch_from_txt()  ProcessPoolExecutor  workers=8  =>  48.7 sec  [OK]"),
    ("STEP 3  Manifest",
     f"_rebuild_manifest.py  < 1 sec  =>  {_n_mani} cases registered  [OK]  (backup: .json.bak)"),
    ("STEP 4  Training",
     "docker exec motor_compare  python train_meshgraphnet.py  --data-dir /workspace/doe_data"),
]
for i,(label,desc) in enumerate(rows):
    y = 2.28 - i*0.42
    t(ax, 1.05, y, label, sz=8.5, bold=True, c="#ffe066", ha="left")
    t(ax, 3.95, y, desc,  sz=8,   c="#dddddd", ha="left")

# ── title ───────────────────────────────────────────────────────────────────
t(ax, 11.0,10.72, "EveryMotor  Phase-1 GNN  Data Pipeline  (2026-04-20)",
  sz=14.5, c=C["host"], bold=True)

plt.tight_layout(pad=0)
_out = Path(r"D:\KDH\NvidiaNemo") / "pipeline_overview.png"
fig.savefig(_out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {_out}")


In [ ]:

# 0-B) DOE 설정 셀 (Host Motor-CAD)
import json
import os
import sys
import importlib
from pathlib import Path



import ansys.motorcad.core as pymotorcad

ROOT = Path.cwd()
EMACH_ROOT = ROOT / "eMach"
if str(EMACH_ROOT) not in sys.path:
    sys.path.insert(0, str(EMACH_ROOT))

from tools.pyutils.sweep import DOEAxis, DOEPoint, build_doe_lhs
import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import (
    doe_batch_run,
    doe_h5_batch_from_txt,
    build_train_manifest,   # Step 3: train_manifest.json 생성 (독립 실행 가능)
)

# ----- 사용자 설정 -----
BASE_MOT = r"D:\KDH\Sim_4SolverX\TestCAD1.mot"
DOE_OUT = Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData")
DOE_OUT.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 40
LHS_SEED = 42
LHS_CRITERION = "maximin"

# Step 1: DOE solve + txt export
PHASES = ["solve", "export_txt"]
# legacy (txt + h5 한 번에): PHASES = ["solve", "export"]

FIRST_STEP = 1
FINAL_STEP = 45
MAG_H5_MESH_COORDS = "by_step_moving_nodes"
MAG_COLUMNS = "RegCode,Bx,By,A,J,Je"
PLOT_MODE = "none"

# 병렬 실행 워커 수 (1=직렬, 2 이상=병렬)
PARALLEL_WORKERS = 8

# 안전 스위치
RUN_DOE = False        # Step 1 (DOE solve + txt)
RUN_H5_EXPORT = True   # Step 2 (txt → h5) + Step 3 (train_manifest.json 자동)

print(f"ROOT={ROOT}")
print(f"EMACH_ROOT={EMACH_ROOT}")
print(f"BASE_MOT={BASE_MOT}")
print(f"DOE_OUT={DOE_OUT}")
print(f"PHASES={PHASES}, N_SAMPLES={N_SAMPLES}, RUN_DOE={RUN_DOE}")
print(f"RUN_H5_EXPORT={RUN_H5_EXPORT}")
print(f"PARALLEL_WORKERS={PARALLEL_WORKERS}")


In [ ]:

# 0-C) 실행 셀: DOE 데이터 생성 (solve + txt-only)
#
# [첫 실행]
#   → Motor-CAD 접속 → build_doe_lhs → doe_grid.json 저장 → 전체 케이스 실행
#
# [재실행 — 0-E-1 통합]
#   → doe_grid.json 로드 (build_doe_lhs 생략)
#   → doe_scan_status 로 상태 점검
#   → needs_solve / not-started 케이스만 doe_batch_run
#   → needs_txt 케이스는 doe_repair_missing_txt 로 txt 재내보내기 (병렬 지원)
#   → 모두 complete 이면 신규 실행 없이 요약만 출력

# ── helpers ──────────────────────────────────────────────────────────────────
def _get_var(mc, name):
    """Motor-CAD get_variable 반환값 언래핑.
    enable_success_variable=True 환경에서는 (returncode, value) 튜플을 반환하므로
    value만 추출합니다."""
    v = mc.get_variable(name)
    return v[1] if isinstance(v, tuple) and len(v) == 2 and isinstance(v[0], int) else v


def _save_doe_grid(grid, path, meta=None):
    """DOEPoint 리스트를 JSON으로 직렬화해 저장합니다."""
    data = {
        "meta": meta or {},
        "points": [
            {"geometry": pt.geometry, "electrical": pt.electrical, "index": pt.index}
            for pt in grid
        ],
    }
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def _load_doe_grid(path):
    """JSON에서 DOEPoint 리스트를 복원합니다."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return [
        DOEPoint(geometry=p["geometry"], electrical=p["electrical"], index=p["index"])
        for p in data["points"]
    ]

# ── 실행 본체 ────────────────────────────────────────────────────────────────
DOE_GRID_JSON = DOE_OUT / "doe_grid.json"

if not RUN_DOE:
    print("RUN_DOE=False 입니다. 설정 확인 후 True로 바꿔 실행하세요.")
else:
    importlib.reload(_doe_batch)
    from tools.motorCAD.pyMCAD.doe_batch import (
        doe_batch_run, doe_scan_status, doe_repair_missing_txt,
    )

    # ── Step 1: DOE 그리드 로드 / 신규 생성 ─────────────────────────────
    if DOE_GRID_JSON.exists():
        doe_grid = _load_doe_grid(DOE_GRID_JSON)
        print(f"[0-C] 기존 DOE 그리드 로드: {len(doe_grid)}개 포인트 ({DOE_GRID_JSON.name})")
        print("      → build_doe_lhs 생략 (Motor-CAD 접속 지연)")
    else:
        # 기준값 읽기용 Motor-CAD 접속
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e0:
            print(f"[warn] existing instance connect failed: {_e0}")
            print("Motor-CAD: launching new instance...")
            mc = pymotorcad.MotorCAD(open_new_instance=True)

        mc.load_from_file(BASE_MOT)
        rb_base = float(_get_var(mc, "Ratio_Bore"))
        rsd_base = float(_get_var(mc, "Ratio_SlotDepth_ParallelSlot"))

        all_axes = [
            DOEAxis("Ratio_Bore", round(rb_base * 0.90, 4), round(rb_base * 1.10, 4), steps=0),
            DOEAxis("Ratio_SlotDepth_ParallelSlot", round(rsd_base * 0.85, 4), round(rsd_base * 1.15, 4), steps=0),
            DOEAxis("PeakCurrent", 10.0, 650.53, steps=0),
            DOEAxis("PhaseAdvance", 0.0, 90.0, steps=0),
        ]
        doe_grid = build_doe_lhs(
            axes=all_axes, n_samples=N_SAMPLES, seed=LHS_SEED, criterion=LHS_CRITERION,
        )
        _save_doe_grid(doe_grid, DOE_GRID_JSON, meta={
            "n_samples": N_SAMPLES, "seed": LHS_SEED, "criterion": LHS_CRITERION,
            "axes": [{"name": ax.name, "min": ax.min_val, "max": ax.max_val} for ax in all_axes],
        })
        print(f"[0-C] 신규 DOE 그리드 생성: {len(doe_grid)}개 포인트 → {DOE_GRID_JSON.name}")

    # ── Step 2: 상태 점검 (0-E-1 통합) ──────────────────────────────────
    scan = doe_scan_status(DOE_OUT, verbose=True)
    _scanned_idx = {c["index"] for c in scan["cases"]}
    _not_started = [pt.index for pt in doe_grid if pt.index not in _scanned_idx]
    _solve_idx = _not_started + scan["solve_needed"]  # 미시작 + 해석 미완료
    _repair_idx = scan["repair_needed"]  # mes 있음, txt 없음

    print(json.dumps({
        "summary": scan["summary"],
        "not_started": len(_not_started),
        "solve_needed": len(_solve_idx),
        "repair_needed": len(_repair_idx),
    }, indent=2, ensure_ascii=False))

    if not _solve_idx and not _repair_idx:
        # 모든 케이스 완료 — 신규 실행 없음
        print("✓ 모든 케이스가 완료 상태입니다. 신규 실행 없음.")
        _txt_ready = scan["summary"].get("complete", 0)
        manifest = {}
    else:
        # Motor-CAD 인스턴스 확보 (아직 없으면 접속)
        if "mc" not in globals():
            try:
                mc = pymotorcad.MotorCAD(open_new_instance=False)
                print("Motor-CAD: existing instance connected")
            except Exception as _e1:
                print(f"[warn] {_e1}")
                mc = pymotorcad.MotorCAD(open_new_instance=True)
                print("Motor-CAD: launching new instance...")
            mc.load_from_file(BASE_MOT)

        # ── Step 3-a: 미시작 / needs_solve 케이스 해석 ───────────────
        if _solve_idx:
            _solve_set = set(_solve_idx)
            _solve_grid = [pt for pt in doe_grid if pt.index in _solve_set]
            print(f"[0-C] solve → {len(_solve_grid)}개 케이스 실행...")
            manifest = doe_batch_run(
                mc, _solve_grid,
                base_mot=BASE_MOT, doe_out_root=DOE_OUT, phases=PHASES,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
            )
        else:
            manifest = {}

        # ── Step 3-b: needs_txt 케이스 txt 재내보내기 ────────────────
        if _repair_idx:
            print(
                f"[0-C] needs_txt 복구 → {len(_repair_idx)}개 케이스 "
                f"(parallel_workers={PARALLEL_WORKERS})..."
            )
            _repair = doe_repair_missing_txt(
                mc, DOE_OUT, case_indices=_repair_idx,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
                verbose=True,
            )
            print(json.dumps({
                "repair_repaired": _repair["repaired"],
                "repair_failed": _repair["failed"],
            }, indent=2, ensure_ascii=False))

        # 최종 완료 수 재집계
        _final_scan = doe_scan_status(DOE_OUT, verbose=False)
        _txt_ready = _final_scan["summary"].get("complete", 0)

    print(json.dumps({
        "base_mot": BASE_MOT,
        "doe_out": str(DOE_OUT),
        "doe_grid_json": str(DOE_GRID_JSON),
        "n_cases": len(doe_grid),
        "txt_ready_cases": _txt_ready,
        "failed": len(manifest.get("failed", [])),
    }, indent=2, ensure_ascii=False))


### 0-D) 선택 셀: TXT → H5 변환 + train_manifest.json 생성

학습/추론 섹션으로 바로 이어갈 때만 실행합니다.

- 실행 환경: 호스트 Python만 필요 (Motor-CAD 불필요)
- 입력: 0-C에서 생성된 `case_NNNN/postproc/Mag_*.txt`
- 출력: `case_NNNN/postproc/Mag_*.h5` → `build_train_manifest()` 자동 호출 → `train_manifest.json` 생성
- `doe_manifest.json` 은 **수정되지 않습니다** (DOE 파라미터 전용, 불변)

> 변환 완료 후 `train_manifest.json` 만 있으면 GNN 학습을 바로 시작할 수 있습니다.  
> `build_train_manifest()` 는 언제든 단독 재실행 가능 (디스크 스캔 기반, 빠름).


In [ ]:

# 0-D) TXT → H5 변환 + train_manifest.json 자동 생성
#
# doe_h5_batch_from_txt()  → case_NNNN/postproc/*.h5 생성
#                          → build_train_manifest() 자동 호출
#                          → train_manifest.json 생성 (디스크 스캔 기반)
# doe_manifest.json 은 수정하지 않습니다.

H5_PARALLEL_WORKERS = 20   # 권장: 4~12 (디스크 I/O 포화 전 코어 수)

if not RUN_H5_EXPORT:
    print("RUN_H5_EXPORT=False 입니다. H5가 필요할 때만 True로 바꿔 실행하세요.")
else:
    import importlib
    import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
    importlib.reload(_doe_batch)
    from tools.motorCAD.pyMCAD.doe_batch import doe_h5_batch_from_txt

    _doe_out = globals().get("DOE_OUT")
    _mesh_coords = globals().get("MAG_H5_MESH_COORDS", "by_step_moving_nodes")
    _n_workers = max(1, int(H5_PARALLEL_WORKERS))

    print(f"[h5 batch] workers={_n_workers}, mesh_coords={_mesh_coords}")

    result = doe_h5_batch_from_txt(
        _doe_out,
        mag_h5_mesh_coords=_mesh_coords,
        verbose=(_n_workers == 1),
        parallel_workers=_n_workers,
        build_train_manifest_after=True,   # train_manifest.json 자동 재생성
    )

    h5_ready = sum(1 for r in result["cases"] if "error" not in r)
    h5_failed = [r for r in result["cases"] if "error" in r]

    print(json.dumps({
        "doe_out": str(_doe_out),
        "parallel_workers": _n_workers,
        "total_cases": len(result["cases"]),
        "h5_ready_cases": h5_ready,
        "failed_cases": [r["index"] for r in h5_failed],
    }, indent=2, ensure_ascii=False))



### 0-E) 수동 복구: DOE 상태 재점검 + TXT 재내보내기

0-C 재실행만으로 복구되지 않는 경우(예: `mc` 접속 없이 상태만 확인하고 싶을 때)에 사용합니다.

> **일반 복구 흐름:** 0-C 셀 재실행만으로 충분합니다 (`doe_grid.json` 이 있으면 자동 상태 점검 → 미완성 케이스만 실행).

**0-E 점검 항목:**
| 상태 | 의미 | 자동 조치 (0-C 재실행 시) |
|------|------|--------------------------|
| `complete` | `.mes` + `.txt` 모두 있음 | 없음 |
| `needs_txt` | `.mes` 있음, `.txt` 없음 | `doe_repair_missing_txt` 자동 호출 |
| `needs_solve` | `.mes` 없음 (해석 미완료) | `doe_batch_run` 자동 호출 |
| `empty` | 초기화 흔적 없음 | 무시 |

**0-E 수동 실행 순서 (0-C 재실행이 어려울 때):**
1. `0-E-1` 상태 점검 셀 실행 → 복구 대상 인덱스 확인
2. `RUN_REPAIR=True`로 변경 후 `0-E-2` 복구 셀 실행


In [ ]:
# 0-E-1) DOE 상태 점검
# 이 셀은 Motor-CAD 없이 실행 가능합니다.
import json
import importlib
from pathlib import Path

# 0-B 셀을 먼저 실행하지 않은 경우를 위한 독립 경로 설정
_doe_out_check = globals().get("DOE_OUT", Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData"))

import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import doe_scan_status

scan_result = doe_scan_status(_doe_out_check, verbose=True)

# 결과 요약 출력
print(json.dumps({
    "doe_out": str(_doe_out_check),
    "summary": scan_result["summary"],
    "repair_needed_count": len(scan_result["repair_needed"]),
    "solve_needed_count": len(scan_result["solve_needed"]),
    "repair_needed_indices": scan_result["repair_needed"][:20],   # 최대 20개만 표시
    "solve_needed_indices": scan_result["solve_needed"][:20],
}, indent=2, ensure_ascii=False))


In [ ]:

# 0-E-2) TXT 재내보내기 복구 셀 (mes 있음 + txt 없는 케이스)
# 안전 스위치 — True로 변경해야 실행됩니다.
RUN_REPAIR = False

if not RUN_REPAIR:
    print("RUN_REPAIR=False 입니다. 복구할 케이스가 있으면 True로 바꿔 실행하세요.")
    print(f"  복구 대상 케이스: {scan_result.get('repair_needed', [])}")
else:
    import importlib
    import tools.motorCAD.pyMCAD.doe_batch as _doe_batch_repair
    importlib.reload(_doe_batch_repair)
    from tools.motorCAD.pyMCAD.doe_batch import doe_repair_missing_txt

    # Motor-CAD 인스턴스 준비 (0-C에서 mc가 이미 있으면 재사용)
    if "mc" not in globals():
        import ansys.motorcad.core as pymotorcad
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e:
            print(f"[warn] {_e}")
            mc = pymotorcad.MotorCAD(open_new_instance=True)
            print("Motor-CAD: new instance launched")

    # 복구 대상 인덱스 (None = 점검에서 발견된 needs_txt 전체)
    REPAIR_CASE_INDICES = scan_result.get("repair_needed") or None
    REPAIR_PARALLEL_WORKERS = int(globals().get("PARALLEL_WORKERS", 1))

    repair_summary = doe_repair_missing_txt(
        mc,
        _doe_out_check,
        case_indices=REPAIR_CASE_INDICES,
        first_step=globals().get("FIRST_STEP", 1),
        final_step=globals().get("FINAL_STEP", 45),
        mag_columns=globals().get("MAG_COLUMNS", "RegCode,Bx,By,A,J,Je"),
        mag_h5_mesh_coords=globals().get("MAG_H5_MESH_COORDS", "by_step_moving_nodes"),
        plot_mode="none",
        parallel_workers=REPAIR_PARALLEL_WORKERS,
        dry_run=False,
        verbose=True,
    )

    print(json.dumps({
        "doe_out": str(_doe_out_check),
        "parallel_workers": REPAIR_PARALLEL_WORKERS,
        "repaired": repair_summary["repaired"],
        "failed": repair_summary["failed"],
        "skipped": repair_summary["skipped"],
    }, indent=2, ensure_ascii=False))


# Phase 1 Tutorial — SymMGN PBC 전체 검증

이 노트북은 Phase 1 (Static SymMGN with PBC) 완료 증거를 생성합니다.

**핵심 원칙:**
- 로컬 커널은 오케스트레이션/시각화만 수행
- 모델 학습/추론은 `motor_compare` 컨테이너 내부에서 실행

**실행 순서:**
| Part | 내용 | 셀 |
|------|------|----|
| 0 | (선택) DOE 데이터 재생성 + H5 후속 변환 | 0-B ~ 0-D |
| A | 환경 + Docker + GPU | 1-4 |
| B | Contract / PBC / Overfit 검증 | 5-7 |
| C | **PBC 경계 가시화** (학습 전 확인) | 8 |
| D | 3-case DOE smoke test | 9-10 |
| E | **Full 40-case 학습** | 11 |
| F | **Full 40-case 추론 + 시각화** | 12-13 |
| G | Phase 1 완료 Evidence | 14

## 1) 환경 설정 및 라이브러리 임포트

로컬에서는 실행 제어/결과 파싱만 수행합니다.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
CONTAINER_NAME = os.environ.get("PHYSICSNEMO_CONTAINER", "motor_compare")
LOG_DIR = ROOT / "logs" / "tutorial"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"CONTAINER_NAME={CONTAINER_NAME}")
print(f"LOG_DIR={LOG_DIR}")

## 2) Docker 실행 헬퍼

학습/추론/테스트는 모두 컨테이너 내부에서 실행합니다.

In [ ]:
import shutil
from datetime import datetime


def run_local(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"[local] {cmd}")
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, check=check)


def run_docker(cmd: str, check: bool = True, stream: bool = True) -> subprocess.CompletedProcess:
    """Docker exec wrapper.

    stream=True(기본): stdout/stderr를 한 줄씩 실시간 출력하면서
    완료 후 CompletedProcess 형태로 반환한다.
    긴 학습 명령어에서 진행 상황을 즉시 확인할 수 있다.
    """
    payload = f"cd /workspace/app && {cmd}"
    payload = payload.replace("\\", "\\\\").replace('"', '\\"')
    full_cmd = f'docker exec {CONTAINER_NAME} bash -lc "{payload}"'
    print(f"[docker] {cmd}")

    if not stream:
        return run_local(full_cmd, check=check)

    # ── 실시간 스트리밍 실행 ─────────────────────────────────────────────────
    import sys, threading, io

    stdout_lines: list[str] = []
    stderr_lines: list[str] = []

    proc = subprocess.Popen(
        full_cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,          # 줄 단위 버퍼
    )

    def _drain(pipe, store: list[str], prefix: str) -> None:
        for line in pipe:
            store.append(line)
            print(f"{prefix}{line}", end="", flush=True)

    t_out = threading.Thread(target=_drain, args=(proc.stdout, stdout_lines, ""), daemon=True)
    t_err = threading.Thread(target=_drain, args=(proc.stderr, stderr_lines, "[ERR] "), daemon=True)
    t_out.start()
    t_err.start()
    t_out.join()
    t_err.join()
    returncode = proc.wait()

    if check and returncode != 0:
        raise subprocess.CalledProcessError(returncode, full_cmd)

    return subprocess.CompletedProcess(
        args=full_cmd,
        returncode=returncode,
        stdout="".join(stdout_lines),
        stderr="".join(stderr_lines),
    )


def save_log(name: str, cp: subprocess.CompletedProcess) -> Path:
    path = LOG_DIR / name
    text = []
    text.append(f"$ returncode={cp.returncode}\n")
    if cp.stdout:
        text.append("\n[stdout]\n")
        text.append(cp.stdout)
    if cp.stderr:
        text.append("\n[stderr]\n")
        text.append(cp.stderr)
    path.write_text("".join(text), encoding="utf-8")
    print(f"saved: {path}")
    return path


def backup_npz_dir(npz_dir: Path, label: str = "") -> "Path | None":
    """NPZ 결과 디렉토리를 타임스탬프 백업으로 이동한다.

    코드(모델·손실·채널·정규화 등)를 수정한 뒤 추론을 재실행하기 전에
    반드시 호출해 기존 결과를 보존한다.

    백업 경로:
        <npz_dir>/../backups/<dirname>_YYYYMMDD_HHMMSS[_label]/

    Args:
        npz_dir: NPZ 파일이 있는 디렉토리 (Path 또는 str).
        label:   백업 디렉토리 이름에 붙일 짧은 설명 (예: "before_4ch").

    Returns:
        백업 디렉토리 Path (NPZ가 없었으면 None).

    Policy:
        - 학습/추론 코드(contracts.py, motor_dataset.py, infer_*.py 등)를
          수정했다면 재실행 전 이 함수를 먼저 호출한다.
        - checkpoint 파일(.pt)도 교체될 경우 수동으로 같은 backups/ 폴더에
          복사해 둔다.
        - 백업은 덮어쓰지 않는다 (타임스탬프로 고유 디렉토리 생성).
    """
    npz_dir = Path(npz_dir)
    if not npz_dir.exists():
        print(f"[backup] {npz_dir} 없음 — 건너뜀")
        return None

    existing_npz = sorted(npz_dir.glob("*.npz"))
    if not existing_npz:
        print(f"[backup] {npz_dir} 에 NPZ 없음 — 건너뜀")
        return None

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = f"_{label}" if label else ""
    backup_dir = npz_dir.parent / "backups" / f"{npz_dir.name}_{ts}{suffix}"
    backup_dir.mkdir(parents=True, exist_ok=True)

    for f in existing_npz:
        shutil.move(str(f), str(backup_dir / f.name))

    print(f"[backup] {len(existing_npz)}개 NPZ → {backup_dir}")
    return backup_dir


print("helpers ready: run_local / run_docker (streaming) / save_log / backup_npz_dir")


## 3) GPU-enabled Docker Compose 기동

GPU, IPC, ulimit 설정이 적용되도록 컨테이너를 재생성합니다.

In [ ]:
cp_compose_up = run_local("docker compose up -d --force-recreate", check=False)
save_log("00_compose_up.log", cp_compose_up)
print(cp_compose_up.stdout)
if cp_compose_up.stderr.strip():
    print(cp_compose_up.stderr)
if cp_compose_up.returncode != 0:
    raise RuntimeError("GPU-enabled docker compose up 에 실패했습니다. compose 설정과 Docker GPU 런타임을 확인하세요.")

## 4) GPU / PyTorch 확인 (컨테이너 내부)

In [ ]:
cp_nvidia_smi = run_docker("nvidia-smi", check=False)
save_log("01_nvidia_smi.log", cp_nvidia_smi)
print(cp_nvidia_smi.stdout or cp_nvidia_smi.stderr)

cp_torch = run_docker(
    "python -c \"import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('cuda_version', torch.version.cuda)\"",
    check=False,
 )
save_log("01_torch_env.log", cp_torch)
print(cp_torch.stdout)
if cp_torch.stderr.strip():
    print(cp_torch.stderr)
if cp_nvidia_smi.returncode != 0 or cp_torch.returncode != 0 or "cuda False" in cp_torch.stdout:
    raise RuntimeError(
        "GPU가 컨테이너에 노출되지 않았습니다. 01_nvidia_smi.log 와 01_torch_env.log를 확인하세요."
    )

## 5) Contract / PBC / Overfit 검증 게이트

Phase 1 계약 경계 테스트 → PBC 경계/계약 호스트 테스트 → Overfit-Single 게이트 → PBC bundle 테스트

In [ ]:
cp_contract = run_docker(
    "PYTHONPATH=/workspace/app pytest -q tests/test_phase1_contract_boundaries.py",
    check=False,
 )
save_log("02_contract_tests.log", cp_contract)
print(cp_contract.stdout)
if cp_contract.stderr.strip():
    print(cp_contract.stderr)
if cp_contract.returncode != 0:
    raise RuntimeError("Contract 경계 테스트 실패. 로그(02_contract_tests.log)를 확인하세요.")

In [ ]:
import subprocess

# torch 없이 돌아가는 경계/계약 테스트는 호스트 venv에서 허용
# (phase1_static.pbc_boundary, pbc_contracts 는 scipy만 필요)
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_pbc_host_tests = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_pbc_host_tests.stdout)
if cp_pbc_host_tests.returncode != 0:
    print("[stderr]", cp_pbc_host_tests.stderr[:600])
print("exit_code:", cp_pbc_host_tests.returncode)

In [ ]:
import subprocess, json, textwrap

OVERFIT_SCRIPT = textwrap.dedent("""
import sys, json, tempfile, subprocess
import numpy as np
from pathlib import Path

try:
    angles = np.deg2rad(np.array([0.0, 22.5, 45.0], dtype=np.float64))
    inner = np.stack([np.cos(angles), np.sin(angles)], axis=1).astype(np.float32)
    pos = inner[None]
    node_type = np.ones((1, 3, 1), dtype=np.float32)
    edges = np.array([[0, 1], [1, 2], [2, 0], [1, 0], [2, 1], [0, 2]], dtype=np.int64)
    ie = edges.T[None]
    pe = np.zeros((1, 2, 0), dtype=np.int64)
    pa = np.zeros((1, 0, 1), dtype=np.float32)
    y = np.random.RandomState(42).randn(1, 3, 5).astype(np.float32)

    with tempfile.TemporaryDirectory() as td:
        npz_path = Path(td) / "overfit.npz"
        np.savez(
            npz_path,
            pos=pos,
            node_type_onehot=node_type,
            interior_edge_index=ie,
            pbc_edge_index=pe,
            pbc_edge_attr=pa,
            y=y,
        )

        r = subprocess.run(
            [sys.executable, "-m", "phase1_static.train",
             "--input-format", "npz", "--data", str(npz_path),
             "--overfit-single", "--epochs", "150",
             "--lr", "1e-2", "--hidden-dim", "32",
             "--seed", "42", "--overfit-target", "1e-2"],
            capture_output=True, text=True, cwd="/workspace/app",
        )
        combined = r.stdout + r.stderr
        epoch_lines = [l for l in combined.splitlines() if "epoch=" in l]
        overfit_lines = [l for l in combined.splitlines() if "Overfit" in l or "overfit" in l]
        print(json.dumps({
            "returncode": r.returncode,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "overfit_gate": overfit_lines[-1] if overfit_lines else "",
        }))
except Exception as exc:
    import traceback
    print(json.dumps({"error": str(exc), "tb": traceback.format_exc()[-400:]}))
""")

cp_overfit = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", OVERFIT_SCRIPT],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)

print("=== Overfit-Single (Docker) ===")
for line in cp_overfit.stdout.splitlines():
    try:
        d = json.loads(line)
        print(json.dumps(d, indent=2, ensure_ascii=False))
    except Exception:
        print(line)
print("exit_code:", cp_overfit.returncode)

In [ ]:
import subprocess

# pbc_bundle 보강 테스트 + 기존 PBC 테스트 전체 확인
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_all_pbc = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_bundle.py",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
        "tests/test_phase1_pbc_pairing.py",
        "tests/test_phase1_pbc_prior.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_all_pbc.stdout)
if cp_all_pbc.returncode != 0:
    print("[stderr]", cp_all_pbc.stderr[:600])
print("exit_code:", cp_all_pbc.returncode)

## 8) PBC 경계 가시화 — 학습 전 확인

실제 DOE 메쉬에서 추출한 master/slave 경계선과 PBC 엣지를 시각화합니다.
학습 전에 PBC topology가 올바르게 구성되었는지 **반드시** 눈으로 확인합니다.

표시 내용:
- 전체 노드 분포 (1/8 섹터)
- master / slave boundary line
- 실제 pbc_edge overlay (anti-periodic, edge_attr=-1.0)
- case별 match ratio, pair 수, group labels

In [ ]:
import importlib
import json

from IPython.display import display

pbc_boundary_module = importlib.import_module(
    "phase1_static.pbc_boundary"
 )
pbc_candidate_module = importlib.import_module(
    "postproc_interop.model.PBCBoundaryCandidate"
 )
pbc_case_module = importlib.import_module(
    "postproc_interop.model.PBCVisualizationCase"
 )
pbc_module = importlib.import_module("postproc_interop.pbc")
pbc_bridge_module = importlib.import_module(
    "postproc_interop.bridges.MotorCADPBCVisualizationBridge"
 )

for module in (
    pbc_boundary_module,
    pbc_candidate_module,
    pbc_case_module,
    pbc_module,
    pbc_bridge_module,
 ):
    importlib.reload(module)

MotorCADPBCVisualizationBridge = pbc_bridge_module.MotorCADPBCVisualizationBridge

PBC_VIS_CASE_INDICES = list(globals().get("DOE_CASE_INDICES", [0, 1])[:2])
if len(PBC_VIS_CASE_INDICES) < 2:
    PBC_VIS_CASE_INDICES = [0, 1]

PBC_VIS_SOURCE_TYPES = list(globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"]))
PBC_VIS_OUT_DIR = ROOT / "results" / "pbc_case_vis"

pbc_bridge = MotorCADPBCVisualizationBridge(ROOT)
pbc_cases, skipped_cases = pbc_bridge.collect_cases(
    case_indices=PBC_VIS_CASE_INDICES,
    source_file_types=PBC_VIS_SOURCE_TYPES,
    data_dir=ROOT / "doe_data",
    output_dir=PBC_VIS_OUT_DIR,
)

summary = {
    "case_indices": [int(case.case_idx) for case in pbc_cases],
    "source_file_types": sorted(
        {case.source_file_type for case in pbc_cases}
    ),
    "pairs": {
        str(case.case_idx): int(case.pbc_forward_index.shape[1])
        for case in pbc_cases
    },
    "groups": {
        str(case.case_idx): list(case.group_labels)
        for case in pbc_cases
    },
    "skipped_cases": skipped_cases,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))
display(
    pbc_bridge.build_case_selector_widget(
        cases=pbc_cases,
        default_group="all",
        default_show_nodes=False,
        default_show_pbc_edges=False,
    )
)

## 9) SymMGN 3-case DOE Smoke Test

Full training 전에 3개 case로 파이프라인이 올바르게 작동하는지 확인합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | 3개 (DOE index 0, 1, 2) |
| Source file type | OnLoadTorque |
| Max steps/case | 2 |
| Epochs | 2 |

In [ ]:
import json

DOE_CASE_INDICES = [0, 1, 2]
DOE_SOURCE_FILE_TYPES = ["OnLoadTorque"]
DOE_MAX_STEPS_PER_CASE = 2
DOE_EPOCHS = 2
DOE_BATCH_SIZE = 2
DOE_HIDDEN_DIM = 64
DOE_SEED = 42

SYM_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_onloadtorque.pt"
SYM_TRAIN_LOG = "11_symm_train_doe.log"

train_cmd = " ".join(
    [
        "python -m phase1_static.train",
        "--input-format doe",
        "--data-dir /workspace/doe_data",
        "--case-indices " + " ".join(str(idx) for idx in DOE_CASE_INDICES),
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps-per-case {DOE_MAX_STEPS_PER_CASE}",
        f"--epochs {DOE_EPOCHS}",
        f"--batch-size {DOE_BATCH_SIZE}",
        f"--hidden-dim {DOE_HIDDEN_DIM}",
        f"--seed {DOE_SEED}",
        f"--ckpt-out results/{SYM_CKPT_PATH.name}",
    ]
)

cp_symm_train = run_docker(train_cmd, check=False)
save_log(SYM_TRAIN_LOG, cp_symm_train)

combined_train = (cp_symm_train.stdout or "") + "\n" + (cp_symm_train.stderr or "")
epoch_lines = [line for line in combined_train.splitlines() if "epoch=" in line]
pbc_skip_lines = [line for line in combined_train.splitlines() if "PBC boundary match skipped" in line]

print("=== SymMGN DOE 학습 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_train.returncode,
            "case_indices": DOE_CASE_INDICES,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "pbc_skip_count": len(pbc_skip_lines),
            "ckpt_saved": SYM_CKPT_PATH.exists(),
            "ckpt_size_kb": round(SYM_CKPT_PATH.stat().st_size / 1024, 1) if SYM_CKPT_PATH.exists() else 0,
            "log_file": str(LOG_DIR / SYM_TRAIN_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_train.returncode != 0:
    print("=== train stderr tail ===")
    print("\n".join(combined_train.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 학습 실패. 11_symm_train_doe.log를 확인하세요.")


## 10) Smoke Test 추론 + 시각화

3-case smoke 모델로 case 0의 첫 step을 추론하고 GT vs Pred 시각화를 확인합니다.

In [ ]:
import json

DOE_SOURCE_FILE_TYPES = globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"])
SYM_CKPT_PATH = globals().get("SYM_CKPT_PATH", ROOT / "results" / "symm_mgn_doe_onloadtorque.pt")
DOE_INFER_CASE_IDX = globals().get("DOE_CASE_INDICES", [0])[0]
DOE_INFER_MAX_STEPS = 1
SYM_INFER_NPZ_PATH = ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz"
SYM_INFER_LOG = "12_symm_infer_doe.log"

infer_cmd = " ".join(
    [
        "python infer_phase1_pbc.py",
        f"--ckpt /workspace/app/results/{SYM_CKPT_PATH.name}",
        "--data-dir /workspace/doe_data",
        f"--case-idx {DOE_INFER_CASE_IDX}",
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps {DOE_INFER_MAX_STEPS}",
        "--batch-size 1",
        f"--out /workspace/app/results/{SYM_INFER_NPZ_PATH.name}",
    ]
)

cp_symm_infer = run_docker(infer_cmd, check=False)
save_log(SYM_INFER_LOG, cp_symm_infer)

combined_infer = (cp_symm_infer.stdout or "") + "\n" + (cp_symm_infer.stderr or "")
metric_lines = [line for line in combined_infer.splitlines() if "RMSE=" in line]

print("=== SymMGN DOE 추론 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_infer.returncode,
            "case_idx": DOE_INFER_CASE_IDX,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "max_steps": DOE_INFER_MAX_STEPS,
            "infer_npz_exists": SYM_INFER_NPZ_PATH.exists(),
            "infer_npz_size_kb": round(SYM_INFER_NPZ_PATH.stat().st_size / 1024, 1) if SYM_INFER_NPZ_PATH.exists() else 0,
            "metric_lines": metric_lines[-5:],
            "log_file": str(LOG_DIR / SYM_INFER_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_infer.returncode != 0:
    print("=== infer stderr tail ===")
    print("\n".join(combined_infer.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 추론 실패. 12_symm_infer_doe.log를 확인하세요.")


In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

DOE_INFER_CASE_IDX = globals().get("DOE_INFER_CASE_IDX", 0)
SYM_INFER_NPZ_PATH = globals().get(
    "SYM_INFER_NPZ_PATH",
    ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz",
)
SYM_VIS_PNG_PATH = ROOT / "logs" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_gt_vs_pred.png"

if not SYM_INFER_NPZ_PATH.exists():
    raise FileNotFoundError(f"추론 NPZ가 없습니다: {SYM_INFER_NPZ_PATH}")

arr = np.load(SYM_INFER_NPZ_PATH, allow_pickle=True)
pos_x = arr["pos_x"]
pos_y = arr["pos_y"]
channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]

fig, axes = plt.subplots(2, len(channels), figsize=(20, 7))
fig.suptitle(
    f"SymMGN DOE case {DOE_INFER_CASE_IDX:04d} - OnLoadTorque GT vs Prediction",
    fontsize=12,
)

for col_idx, (channel_name, label) in enumerate(zip(channels, labels)):
    gt_values = arr[f"gt_{channel_name}"]
    pred_values = arr[f"pred_{channel_name}"]
    vmin = float(min(gt_values.min(), pred_values.min()))
    vmax = float(max(gt_values.max(), pred_values.max()))

    gt_plot = axes[0, col_idx].scatter(
        pos_x,
        pos_y,
        c=gt_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(gt_plot, ax=axes[0, col_idx], fraction=0.04)

    pred_plot = axes[1, col_idx].scatter(
        pos_x,
        pos_y,
        c=pred_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(pred_plot, ax=axes[1, col_idx], fraction=0.04)

plt.tight_layout()
SYM_VIS_PNG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SYM_VIS_PNG_PATH, dpi=110, bbox_inches="tight")
plt.show()
plt.close()

meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
metrics = json.loads(arr["metrics"].item()) if "metrics" in arr.files else {}

print(
    json.dumps(
        {
            "saved_png": str(SYM_VIS_PNG_PATH),
            "png_size_kb": round(SYM_VIS_PNG_PATH.stat().st_size / 1024, 1),
            "meta": meta,
            "metrics": metrics,
        },
        indent=2,
        ensure_ascii=False,
    )
)

# 학습전 원데이터 import후 데이터포맷별 시각화

In [ ]:
# ── Cell VIZ-0: 샘플 로드 + region 맵 준비 (v2: SB 포함 완전 렌더링) ──────
# numpy + h5py only (torch 불필요 — 컨테이너 전용)
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, r"D:\KDH\NvidiaNemo")

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import h5py, json, pathlib

# ── 파라미터 ──────────────────────────────────────────────────────────────────
DOE_DATA_DIR = r"D:\KDH\Sim_4SolverX\DOE4TrainingData"
CASE_IDX = 17          # 시각화할 케이스 인덱스
STEP_IDX = 31          # H5 step 인덱스 (0 = 첫 번째 step)
SOURCE_TYPE = "OnLoadTorque"

# ── H5 경로 ───────────────────────────────────────────────────────────────────
manifest = json.loads(pathlib.Path(f"{DOE_DATA_DIR}/doe_manifest.json").read_text())
case = next(c for c in manifest["cases"] if c["index"] == CASE_IDX)
h5_path = next(
    p for p in case["h5_paths"]
    if SOURCE_TYPE.lower() in pathlib.Path(p).name.lower().replace("\\", "/")
)
h5_local = pathlib.Path(
    str(h5_path)
    .replace("D:\\KDH\\Sim_4SolverX\\DOE_TrainingData", DOE_DATA_DIR)
    .replace("/", "\\")
)
if not h5_local.exists():
    h5_local = pathlib.Path(DOE_DATA_DIR) / f"case_{CASE_IDX:04d}" / "postproc" / pathlib.Path(h5_path).name

# v2 H5 우선 사용 (SB 노드 좌표 + a_node 포함)
h5_v2 = h5_local.parent / (h5_local.stem + "_v2.h5")
if h5_v2.exists():
    h5_local = h5_v2
print(f"H5 파일: {h5_local}")
assert h5_local.exists(), f"H5 파일을 찾을 수 없습니다: {h5_local}"

# ── H5 파싱 ──────────────────────────────────────────────────────────────────
with h5py.File(h5_local, "r") as f:
    steps        = np.asarray(f["steps"][:] if "steps" in f else [int(f["step"][()])], dtype=np.int32)
    node_id      = np.asarray(f["mesh/node_id"][:], dtype=np.int32)
    node_x0      = np.asarray(f["mesh/node_x_mm"][:], dtype=np.float64)
    node_y0      = np.asarray(f["mesh/node_y_mm"][:], dtype=np.float64)
    raw_n1       = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
    raw_n2       = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
    raw_n3       = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
    reg_code_raw = np.asarray(f["mesh/reg_code"][:], dtype=np.int32)
    moving_reg_codes = (
        np.asarray(f["mesh/moving_reg_codes"][:], dtype=np.int32)
        if "mesh/moving_reg_codes" in f else np.array([], dtype=np.int32)
    )
    moving_idx = (
        np.asarray(f["mesh/moving_node_indices"][:], dtype=np.int32)
        if "mesh/moving_node_indices" in f else None
    )
    x_bsm = np.asarray(f["mesh/node_x_mm_by_step_moving"][:], dtype=np.float64) if "mesh/node_x_mm_by_step_moving" in f else None
    y_bsm = np.asarray(f["mesh/node_y_mm_by_step_moving"][:], dtype=np.float64) if "mesh/node_y_mm_by_step_moving" in f else None
    x_bs  = np.asarray(f["mesh/node_x_mm_by_step"][:], dtype=np.float64) if "mesh/node_x_mm_by_step" in f else None
    y_bs  = np.asarray(f["mesh/node_y_mm_by_step"][:], dtype=np.float64) if "mesh/node_y_mm_by_step" in f else None
    reg_codes_meta = np.asarray(f["regions/reg_code"][:], dtype=np.int32)
    reg_names_meta = [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in f["regions/name"][:]]
    REGION_NAME    = {int(c): str(n) for c, n in zip(reg_codes_meta, reg_names_meta)}

    # Element fields (n_steps × n_elements)
    bx_all = np.asarray(f["fields/bx"][:], dtype=np.float32)
    by_all = np.asarray(f["fields/by"][:], dtype=np.float32)
    a_all  = np.asarray(f["fields/a"][:],  dtype=np.float32) if "fields/a"  in f else np.zeros_like(bx_all)
    j_all  = np.asarray(f["fields/j"][:],  dtype=np.float32) if "fields/j"  in f else np.zeros_like(bx_all)
    je_all = np.asarray(f["fields/je"][:], dtype=np.float32) if "fields/je" in f else np.zeros_like(bx_all)

    # Node-level A (raw FEM primary solution from NodesTable)
    a_node_all = np.asarray(f["fields/a_node"][:], dtype=np.float32) if "fields/a_node" in f else None

    # ── slideband 메타 ────────────────────────────────────────────────────
    if "slideband/reg_codes" in f:
        SLIDEBAND_CODES = set(np.asarray(f["slideband/reg_codes"][:], dtype=np.int32).tolist())
    else:
        SLIDEBAND_CODES = set()

    # Per-step SB element connectivity (CSR)
    _has_sb_conn = "slideband/offsets" in f and len(SLIDEBAND_CODES) > 0
    if _has_sb_conn:
        sb_offsets = np.asarray(f["slideband/offsets"][:], dtype=np.int64)
        sb_n1_all  = np.asarray(f["slideband/node_1"][:], dtype=np.int32)
        sb_n2_all  = np.asarray(f["slideband/node_2"][:], dtype=np.int32)
        sb_n3_all  = np.asarray(f["slideband/node_3"][:], dtype=np.int32)
        sb_rc_all  = np.asarray(f["slideband/reg_code"][:], dtype=np.int32)
        sb_bx_all  = np.asarray(f["slideband/bx"][:], dtype=np.float32)
        sb_by_all  = np.asarray(f["slideband/by"][:], dtype=np.float32)
        sb_a_all   = np.asarray(f["slideband/a"][:],  dtype=np.float32)
        sb_j_all   = np.asarray(f["slideband/j"][:],  dtype=np.float32)
        sb_je_all  = np.asarray(f["slideband/je"][:], dtype=np.float32)
    else:
        sb_offsets = None

    # Per-step SB extra node coordinates (CSR)
    _has_sb_nodes = "slideband/node_offsets" in f
    if _has_sb_nodes:
        sb_node_offsets = np.asarray(f["slideband/node_offsets"][:], dtype=np.int64)
        sb_node_ids_all = np.asarray(f["slideband/node_id"][:], dtype=np.int32)
        sb_node_x_all   = np.asarray(f["slideband/node_x_mm"][:], dtype=np.float32)
        sb_node_y_all   = np.asarray(f["slideband/node_y_mm"][:], dtype=np.float32)
        sb_node_a_all   = np.asarray(f["slideband/node_a"][:], dtype=np.float32) if "slideband/node_a" in f else None
    else:
        sb_node_offsets = None

# ── node_id → 0-based index LUT ──────────────────────────────────────────────
sorted_ids = np.sort(node_id)
max_nid    = int(node_id.max()) + 1
lut        = np.full(max_nid, -1, dtype=np.int64)
for i, nid in enumerate(sorted_ids):
    lut[nid] = i
sort_order = np.argsort(node_id)
N_base = len(sorted_ids)

# ── triangle 인덱스 변환 (main mesh, step-0 기준) ─────────────────────────────
m = min(len(raw_n1), len(raw_n2), len(raw_n3), len(reg_code_raw))
i1 = lut[np.clip(raw_n1[:m], 0, max_nid - 1)]
i2 = lut[np.clip(raw_n2[:m], 0, max_nid - 1)]
i3 = lut[np.clip(raw_n3[:m], 0, max_nid - 1)]
valid_elem   = (i1 >= 0) & (i2 >= 0) & (i3 >= 0)
i1v, i2v, i3v = i1[valid_elem], i2[valid_elem], i3[valid_elem]
reg_code_elem = reg_code_raw[:m][valid_elem].astype(np.int32)
triangles_np  = np.stack([i1v, i2v, i3v], axis=1)

# Separate static (non-SB) vs SB from main mesh
sb_elem_mask = np.isin(reg_code_elem, list(SLIDEBAND_CODES))
_i1_static = i1v[~sb_elem_mask]
_i2_static = i2v[~sb_elem_mask]
_i3_static = i3v[~sb_elem_mask]
_reg_static = reg_code_elem[~sb_elem_mask]

# ── element field matrices ────────────────────────────────────────────────────
bx_mat = bx_all.reshape(1, -1) if bx_all.ndim == 1 else bx_all
by_mat = by_all.reshape(1, -1) if by_all.ndim == 1 else by_all
a_mat  =  a_all.reshape(1, -1) if  a_all.ndim == 1 else  a_all
j_mat  =  j_all.reshape(1, -1) if  j_all.ndim == 1 else  j_all
je_mat = je_all.reshape(1, -1) if je_all.ndim == 1 else je_all

# ── step별 base 노드 좌표 ────────────────────────────────────────────────────
x = node_x0.copy()
y = node_y0.copy()
if x_bs is not None and y_bs is not None:
    xs, ys = x_bs[STEP_IDX], y_bs[STEP_IDX]
    ok = np.isfinite(xs) & np.isfinite(ys)
    x[ok], y[ok] = xs[ok], ys[ok]
if x_bsm is not None and y_bsm is not None and moving_idx is not None and moving_idx.size > 0:
    n_nodes     = node_id.size
    mov_mask    = (moving_idx >= 0) & (moving_idx < n_nodes)
    xs_mov, ys_mov = x_bsm[STEP_IDX], y_bsm[STEP_IDX]
    n_mov       = min(len(xs_mov), mov_mask.sum())
    valid_mi    = np.where(mov_mask)[0][:n_mov]
    node_indices = moving_idx[valid_mi]
    xv, yv      = xs_mov[valid_mi], ys_mov[valid_mi]
    ok_fin      = np.isfinite(xv) & np.isfinite(yv)
    x[node_indices[ok_fin]] = xv[ok_fin]
    y[node_indices[ok_fin]] = yv[ok_fin]
base_node_x = x[sort_order].astype(np.float64)
base_node_y = y[sort_order].astype(np.float64)

# ── Per-step SB merge: 노드 확장 + 요소 합치기 ───────────────────────────────
ext_lut    = lut.copy()
ext_node_x = base_node_x.copy()
ext_node_y = base_node_y.copy()
N = N_base
N_extra = 0

# Node-level A for base nodes
if a_node_all is not None:
    ext_a_node = a_node_all[STEP_IDX][sort_order].copy()
else:
    ext_a_node = None

# Extend with SB extra nodes for this step
if sb_node_offsets is not None and STEP_IDX < len(sb_node_offsets) - 1:
    sn_s = int(sb_node_offsets[STEP_IDX])
    sn_e = int(sb_node_offsets[STEP_IDX + 1])
    if sn_e > sn_s:
        sb_nids = sb_node_ids_all[sn_s:sn_e]
        sb_nx   = sb_node_x_all[sn_s:sn_e]
        sb_ny   = sb_node_y_all[sn_s:sn_e]
        # Extend LUT for extra node IDs
        new_max_nid = max(int(sb_nids.max()) + 1, len(ext_lut))
        if new_max_nid > len(ext_lut):
            ext_lut = np.concatenate([ext_lut, np.full(new_max_nid - len(ext_lut), -1, dtype=np.int64)])
        for k, nid in enumerate(sb_nids.tolist()):
            ext_lut[nid] = N + k
        ext_node_x = np.concatenate([ext_node_x, sb_nx.astype(np.float64)])
        ext_node_y = np.concatenate([ext_node_y, sb_ny.astype(np.float64)])
        if ext_a_node is not None and sb_node_a_all is not None:
            ext_a_node = np.concatenate([ext_a_node, sb_node_a_all[sn_s:sn_e]])
        N_extra = sn_e - sn_s
        N = N + N_extra

# Build merged SB elements for this step
if _has_sb_conn and STEP_IDX < len(sb_offsets) - 1:
    sb_s = int(sb_offsets[STEP_IDX])
    sb_e = int(sb_offsets[STEP_IDX + 1])
    sb_n1_step = sb_n1_all[sb_s:sb_e]
    sb_n2_step = sb_n2_all[sb_s:sb_e]
    sb_n3_step = sb_n3_all[sb_s:sb_e]
    sb_rc_step = sb_rc_all[sb_s:sb_e]
    # Convert to 0-based extended indices
    sb_i1 = ext_lut[np.clip(sb_n1_step, 0, len(ext_lut) - 1)]
    sb_i2 = ext_lut[np.clip(sb_n2_step, 0, len(ext_lut) - 1)]
    sb_i3 = ext_lut[np.clip(sb_n3_step, 0, len(ext_lut) - 1)]
    sb_ok = (sb_i1 >= 0) & (sb_i2 >= 0) & (sb_i3 >= 0)
    # Merged connectivity: static (non-SB) + this step's SB
    merged_i1  = np.concatenate([_i1_static, sb_i1[sb_ok]])
    merged_i2  = np.concatenate([_i2_static, sb_i2[sb_ok]])
    merged_i3  = np.concatenate([_i3_static, sb_i3[sb_ok]])
    merged_reg = np.concatenate([_reg_static, sb_rc_step[sb_ok]])
    triangles_merged = np.stack([merged_i1, merged_i2, merged_i3], axis=1)
    # Merged element fields: static portion + this step's SB portion
    bx_step = np.concatenate([bx_mat[STEP_IDX][valid_elem][~sb_elem_mask], sb_bx_all[sb_s:sb_e][sb_ok]])
    by_step = np.concatenate([by_mat[STEP_IDX][valid_elem][~sb_elem_mask], sb_by_all[sb_s:sb_e][sb_ok]])
    a_step  = np.concatenate([a_mat[STEP_IDX][valid_elem][~sb_elem_mask],  sb_a_all[sb_s:sb_e][sb_ok]])
    j_step  = np.concatenate([j_mat[STEP_IDX][valid_elem][~sb_elem_mask],  sb_j_all[sb_s:sb_e][sb_ok]])
    je_step = np.concatenate([je_mat[STEP_IDX][valid_elem][~sb_elem_mask], sb_je_all[sb_s:sb_e][sb_ok]])
    _SB_RENDERED = True
else:
    # Fallback: SB 제외 (v1 H5)
    triangles_merged = triangles_np[~sb_elem_mask]
    merged_reg = reg_code_elem[~sb_elem_mask]
    bx_step = bx_mat[STEP_IDX][valid_elem][~sb_elem_mask]
    by_step = by_mat[STEP_IDX][valid_elem][~sb_elem_mask]
    a_step  = a_mat[STEP_IDX][valid_elem][~sb_elem_mask]
    j_step  = j_mat[STEP_IDX][valid_elem][~sb_elem_mask]
    je_step = je_mat[STEP_IDX][valid_elem][~sb_elem_mask]
    _SB_RENDERED = False

# ── Element → Node scatter ────────────────────────────────────────────────────
def elem_to_node_avg(field_elem, n_nodes, tri):
    acc = np.zeros(n_nodes, dtype=np.float64)
    cnt = np.zeros(n_nodes, dtype=np.int32)
    for ch in range(3):
        np.add.at(acc, tri[:, ch], field_elem)
        np.add.at(cnt, tri[:, ch], 1)
    return (acc / np.maximum(cnt, 1)).astype(np.float32)

bx_node = elem_to_node_avg(bx_step, N, triangles_merged)
by_node = elem_to_node_avg(by_step, N, triangles_merged)
j_node  = elem_to_node_avg(j_step,  N, triangles_merged)
je_node = elem_to_node_avg(je_step, N, triangles_merged)

# A: node-level 직접 사용 (FEM primary solution, scatter 불필요)
if ext_a_node is not None and len(ext_a_node) == N:
    a_node = ext_a_node.astype(np.float32)
    _A_SOURCE = "node-level (fields/a_node)"
else:
    a_node = elem_to_node_avg(a_step, N, triangles_merged)
    _A_SOURCE = "element scatter (fallback)"

# ── node_reg, 기타 변수 ──────────────────────────────────────────────────────
node_reg = np.zeros(N, dtype=np.int32)
for e_idx in range(len(triangles_merged)):
    for n_idx in triangles_merged[e_idx]:
        if node_reg[n_idx] == 0:
            node_reg[n_idx] = merged_reg[e_idx]

MOVING_SET = set(moving_reg_codes.tolist())
SLOT_CODES = {c for c, n in REGION_NAME.items() if "armature" in n.lower() or "turn" in n.lower() or "impreg" in n.lower()}

node_x = ext_node_x.astype(np.float32)
node_y = ext_node_y.astype(np.float32)
triang = mtri.Triangulation(node_x, node_y, triangles_merged)

sb_status = f"SB 포함 ({sb_ok.sum()} elem, +{N_extra} nodes)" if _SB_RENDERED else "SB 제외 (v1 fallback)"
print(f"로드 완료 — step={STEP_IDX}/{len(steps)}, nodes={N} (base={N_base}, extra={N_extra})")
print(f"  elements={len(triangles_merged)} ({sb_status})")
print(f"  SLIDEBAND_CODES (H5): {sorted(SLIDEBAND_CODES)}")
print(f"  A source: {_A_SOURCE}")
print(f"  |Bx| max={bx_node.max():.4f} T,  |By| max={by_node.max():.4f} T")
print(f"  |A|  max={a_node.max():.4e} Wb/m,  |Je| max={je_node.max():.4f} A/m²")
print(f"  |j_raw(node)| max={j_node.max():.4f} A/m²")
print(f"  moving region codes: {sorted(MOVING_SET)}")

In [ ]:
# ── Cell VIZ-1: 4채널 필드 맵 (Bx, By, A, Je) + j_raw ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    f"Field Maps — Case {CASE_IDX:04d}, {SOURCE_TYPE}, step={STEP_IDX}",
    fontsize=14, fontweight="bold"
)

FIELDS = [
    (bx_node,  "Bx [T]",     "RdBu_r"),
    (by_node,  "By [T]",     "RdBu_r"),
    (a_node,   "A [Wb/m]",   "plasma"),
    (je_node,  "Je [A/m²]",  "coolwarm"),
    (j_node,   "J_raw [A/m²] (FEM ref)", "coolwarm"),
    (je_node - j_node, "Je − J_raw [A/m²]", "coolwarm"),
]

for ax, (field, label, cmap) in zip(axes.flat, FIELDS):
    vmax = np.abs(field).max() or 1.0
    vmin = -vmax if cmap in ("RdBu_r", "coolwarm") else 0.0
    tc = ax.tripcolor(triang, field, cmap=cmap, vmin=vmin, vmax=vmax, shading="gouraud")
    fig.colorbar(tc, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(label, fontsize=11)
    ax.set_aspect("equal")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(f"viz_fields_case{CASE_IDX:04d}_step{STEP_IDX}.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ 저장: viz_fields_case***.png")


In [ ]:
# ── DEBUG: merged mesh 품질 검증 ─────────────────────────────────────────────
# SB가 포함된 merged 삼각형의 edge 길이 이상치 검사
from collections import Counter

print(f"Mesh 구성: {len(triangles_merged)} elements, {N} nodes (base={N_base}, SB extra={N_extra})")
print(f"SB rendered: {_SB_RENDERED}")
print(f"A source: {_A_SOURCE}")

# 삼각형 edge 길이 통계 (merged mesh 기준)
edge_lens = []
for tri in triangles_merged:
    pts = np.stack([node_x[tri], node_y[tri]], axis=1)
    for a, b in [(0,1),(1,2),(2,0)]:
        edge_lens.append(np.linalg.norm(pts[a] - pts[b]))
edge_lens = np.array(edge_lens)
p99 = np.percentile(edge_lens, 99)
median_len = np.median(edge_lens)
print(f"\nEdge 길이 (merged): median={median_len:.3f}, p99={p99:.3f}, max={edge_lens.max():.3f} mm")
print(f"  ratio max/median = {edge_lens.max()/median_len:.1f}x")

# 길이 이상 삼각형의 region 분포
stretched_mask = np.zeros(len(triangles_merged), dtype=bool)
for idx, tri in enumerate(triangles_merged):
    pts = np.stack([node_x[tri], node_y[tri]], axis=1)
    maxe = max(np.linalg.norm(pts[a]-pts[b]) for a,b in [(0,1),(1,2),(2,0)])
    if maxe > p99 * 2:
        stretched_mask[idx] = True
stretched_regs = merged_reg[stretched_mask]
dist = Counter(stretched_regs.tolist())
print(f"\n늘어진 삼각형 (edge > 2×p99): {stretched_mask.sum()}개")
if stretched_mask.sum() > 0:
    for c, cnt in dist.most_common(10):
        print(f"  code={c:3d}  name={REGION_NAME.get(c,'?'):30s}  count={cnt}")
else:
    print("  ✓ 늘어진 삼각형 없음 — SB 완전 렌더링 정상")

# SB region별 요소 수 확인
print(f"\nSB region 요소 분포:")
sb_in_merged = np.isin(merged_reg, list(SLIDEBAND_CODES))
for c in sorted(SLIDEBAND_CODES):
    cnt = int((merged_reg == c).sum())
    print(f"  code={c:3d}  name={REGION_NAME.get(c,'?'):30s}  elements={cnt}")

In [ ]:
# ── Cell VIZ-1b: Step별 필드 애니메이션 GIF (PyVista offscreen 렌더링) ───
# matplotlib tripcolor(CPU) → PyVista VTK offscreen(GPU) 교체
# scatter: np.add.at 루프 → np.bincount (~10× 빠름)
import pyvista as pv
from io import BytesIO
from PIL import Image
import time

pv.global_theme.allow_empty_mesh = True

# ── 파라미터 ──────────────────────────────────────────────────────────────────
STEP_STRIDE = max(1, len(steps) // 60)
GIF_FPS     = 6
WIN_W, WIN_H = 1200, 900   # PyVista window size (≈ GIF 해상도)

step_indices = list(range(0, len(steps), STEP_STRIDE))
print(f"총 {len(steps)} step 중 {len(step_indices)}프레임 생성 (stride={STEP_STRIDE})")

# ── global colorbar 범위 (SB 필드 포함) ──────────────────────────────────────
_bx_sb = float(np.abs(sb_bx_all).max()) if sb_offsets is not None else 0.0
_by_sb = float(np.abs(sb_by_all).max()) if sb_offsets is not None else 0.0
_a_sb  = float(np.abs(sb_a_all ).max()) if sb_offsets is not None else 0.0
_je_sb = float(np.abs(sb_je_all).max()) if sb_offsets is not None else 0.0
bx_gmax = max(float(np.abs(bx_mat[:, valid_elem][:, ~sb_elem_mask]).max()), _bx_sb) or 1.0
by_gmax = max(float(np.abs(by_mat[:, valid_elem][:, ~sb_elem_mask]).max()), _by_sb) or 1.0
a_gmax  = max(float(np.abs( a_mat[:, valid_elem][:, ~sb_elem_mask]).max()), _a_sb ) or 1.0
je_gmax = max(float(np.abs(je_mat[:, valid_elem][:, ~sb_elem_mask]).max()), _je_sb) or 1.0

field_meta = [
    ("Bx [T]",    "RdBu",   (-bx_gmax, bx_gmax)),
    ("By [T]",    "RdBu",   (-by_gmax, by_gmax)),
    ("A [Wb/m]",  "plasma",  (0.0,      a_gmax )),
    ("Je [A/m²]", "coolwarm",     (0.0,      je_gmax)),
]

# ── bincount 기반 scatter (np.add.at 루프보다 ~10× 빠름) ──────────────────
def fast_scatter(field_elem, tri_flat, n_nodes):
    """Element → node average: bincount 방식, 순수 numpy."""
    w   = np.tile(field_elem.astype(np.float64), 3)
    acc = np.bincount(tri_flat, weights=w, minlength=n_nodes)
    cnt = np.bincount(tri_flat, minlength=n_nodes)
    return (acc / np.maximum(cnt, 1)).astype(np.float32)

# static 삼각형 flat index (루프 밖에서 1회 계산)
_tri_flat_static = np.concatenate([_i1_static, _i2_static, _i3_static])

# ── 프레임 생성 루프 ─────────────────────────────────────────────────────────
frames = []
t0 = time.time()

for frame_i, si in enumerate(step_indices):

    # ── base 노드 좌표 재계산 ─────────────────────────────────────────────
    _x = node_x0.copy()
    _y = node_y0.copy()
    if x_bsm is not None and y_bsm is not None and moving_idx is not None and moving_idx.size > 0:
        _n_nodes = node_id.size
        _mov_mask = (moving_idx >= 0) & (moving_idx < _n_nodes)
        _xs_mov, _ys_mov = x_bsm[si], y_bsm[si]
        _n_mov = min(len(_xs_mov), _mov_mask.sum())
        _valid_mi = np.where(_mov_mask)[0][:_n_mov]
        _node_idx = moving_idx[_valid_mi]
        _xv, _yv  = _xs_mov[_valid_mi], _ys_mov[_valid_mi]
        _ok = np.isfinite(_xv) & np.isfinite(_yv)
        _x[_node_idx[_ok]] = _xv[_ok]
        _y[_node_idx[_ok]] = _yv[_ok]
    _base_nx = _x[sort_order]
    _base_ny = _y[sort_order]

    # ── SB extra nodes 확장 ───────────────────────────────────────────────
    _ext_nx  = _base_nx.copy()
    _ext_ny  = _base_ny.copy()
    _ext_lut = lut.copy()
    _N       = N_base

    if sb_node_offsets is not None and si < len(sb_node_offsets) - 1:
        _sn_s = int(sb_node_offsets[si])
        _sn_e = int(sb_node_offsets[si + 1])
        if _sn_e > _sn_s:
            _sb_nids = sb_node_ids_all[_sn_s:_sn_e]
            _new_max = max(int(_sb_nids.max()) + 1, len(_ext_lut))
            if _new_max > len(_ext_lut):
                _ext_lut = np.concatenate([_ext_lut,
                    np.full(_new_max - len(_ext_lut), -1, dtype=np.int64)])
            for k, nid in enumerate(_sb_nids.tolist()):
                _ext_lut[nid] = _N + k
            _ext_nx = np.concatenate([_ext_nx, sb_node_x_all[_sn_s:_sn_e].astype(np.float64)])
            _ext_ny = np.concatenate([_ext_ny, sb_node_y_all[_sn_s:_sn_e].astype(np.float64)])
            _N += _sn_e - _sn_s

    # ── SB element merge ──────────────────────────────────────────────────
    _bx_s = bx_mat[si][valid_elem][~sb_elem_mask]
    _by_s = by_mat[si][valid_elem][~sb_elem_mask]
    _a_s  =  a_mat[si][valid_elem][~sb_elem_mask]
    _je_s = je_mat[si][valid_elem][~sb_elem_mask]

    if sb_offsets is not None and si < len(sb_offsets) - 1:
        _sb_s = int(sb_offsets[si])
        _sb_e = int(sb_offsets[si + 1])
        _si1 = _ext_lut[np.clip(sb_n1_all[_sb_s:_sb_e], 0, len(_ext_lut) - 1)]
        _si2 = _ext_lut[np.clip(sb_n2_all[_sb_s:_sb_e], 0, len(_ext_lut) - 1)]
        _si3 = _ext_lut[np.clip(sb_n3_all[_sb_s:_sb_e], 0, len(_ext_lut) - 1)]
        _sb_ok = (_si1 >= 0) & (_si2 >= 0) & (_si3 >= 0)
        _i1_m = np.concatenate([_i1_static, _si1[_sb_ok]])
        _i2_m = np.concatenate([_i2_static, _si2[_sb_ok]])
        _i3_m = np.concatenate([_i3_static, _si3[_sb_ok]])
        _bx = np.concatenate([_bx_s, sb_bx_all[_sb_s:_sb_e][_sb_ok]])
        _by = np.concatenate([_by_s, sb_by_all[_sb_s:_sb_e][_sb_ok]])
        _a  = np.concatenate([_a_s,  sb_a_all[_sb_s:_sb_e][_sb_ok]])
        _je = np.concatenate([_je_s, sb_je_all[_sb_s:_sb_e][_sb_ok]])
        _tri_flat = np.concatenate([_i1_m, _i2_m, _i3_m])
        _tri_m = np.stack([_i1_m, _i2_m, _i3_m], axis=1)
    else:
        _tri_flat = _tri_flat_static
        _tri_m = np.stack([_i1_static, _i2_static, _i3_static], axis=1)
        _bx, _by, _a, _je = _bx_s, _by_s, _a_s, _je_s

    # ── bincount scatter ──────────────────────────────────────────────────
    bx_n = fast_scatter(_bx, _tri_flat, _N)
    by_n = fast_scatter(_by, _tri_flat, _N)
    a_n  = fast_scatter(_a,  _tri_flat, _N)
    je_n = fast_scatter(_je, _tri_flat, _N)

    # ── PyVista PolyData 구성 ────────────────────────────────────────────
    pts   = np.column_stack([_ext_nx, _ext_ny, np.zeros(_N)])
    faces = np.column_stack([
        np.full(len(_tri_m), 3, dtype=np.int32), _tri_m.astype(np.int32)
    ]).ravel()
    mesh = pv.PolyData(pts, faces)

    # ── PyVista 4-panel offscreen 렌더링 ─────────────────────────────────
    pl = pv.Plotter(shape=(2, 2), off_screen=True, window_size=[WIN_W, WIN_H])
    pl.set_background("white")

    for idx, (fn, (label, cmap, clim)) in enumerate(
            zip([bx_n, by_n, a_n, je_n], field_meta)):
        row, col = divmod(idx, 2)
        pl.subplot(row, col)
        _m = mesh.copy(deep=False)
        _m.point_data["f"] = fn
        pl.add_mesh(_m, scalars="f", cmap=cmap, clim=list(clim),
                    show_scalar_bar=True,
                    scalar_bar_args={"title": label, "n_labels": 3,
                                     "title_font_size": 10, "label_font_size": 8})
        pl.add_text(f"step={si}  {label}", font_size=7, color="black", position="upper_left")
        pl.view_xy()
        pl.reset_camera()

    img = pl.screenshot(return_img=True)
    pl.close()
    frames.append(Image.fromarray(img))

    if (frame_i + 1) % 5 == 0:
        elapsed = time.time() - t0
        print(f"  {frame_i + 1}/{len(step_indices)} frames  "
              f"({(frame_i+1)/elapsed:.1f} f/s, {elapsed:.0f}s elapsed)")

total_t = time.time() - t0
print(f"\n렌더링 완료: {len(frames)} frames / {total_t:.1f}s  "
      f"({len(frames)/total_t:.1f} f/s, matplotlib 대비 기대 속도향상 3~8×)")

# ── GIF 저장 ─────────────────────────────────────────────────────────────────
gif_path = f"viz_fields_case{CASE_IDX:04d}_anim_pv.gif"
frames[0].save(
    gif_path, save_all=True, append_images=frames[1:],
    duration=int(1000 / GIF_FPS), loop=0,
)
print(f"GIF 저장 완료: {gif_path}  ({len(frames)} frames, {GIF_FPS} fps)")
print(f"   파일 크기: {pathlib.Path(gif_path).stat().st_size / 1e6:.1f} MB")


In [ ]:
# ── Cell VIZ-2: 경계 조건 시각화 ────────────────────────────────────────────
# (a) Region 컬러맵  (b) Dirichlet A=0 외부 경계  (c) Interior edge 밀도
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(f"Mesh & Boundary Conditions — Case {CASE_IDX:04d}, step={STEP_IDX}", fontsize=13, fontweight="bold")

ax = axes[0]
ax.set_title("Region 분류")

REGION_COLOR = {
    "stator":    "#4a90d9",
    "slot":      "#f5a623",
    "rotor":     "#d0021b",
    "magnet":    "#7ed321",
    "slideband": "#9b59b6",
    "shaft":     "#8b572a",
    "air":       "#cccccc",
}

def classify_region(code):
    name = REGION_NAME.get(code, "").lower()
    if "stator" in name and "air" not in name and "wedge" not in name:
        return "stator"
    if "armature" in name or "turn" in name or "impreg" in name:
        return "slot"
    if "rotor" in name or "pocket" in name:
        return "rotor"
    if "magnet" in name:
        return "magnet"
    if code in SLIDEBAND_CODES:
        return "slideband"
    if "shaft" in name:
        return "shaft"
    return "air"

# merged_reg 사용 (static + SB 전체 요소 포함)
elem_class = np.array([classify_region(c) for c in merged_reg])
CLASS_ORDER = ["stator", "slot", "rotor", "magnet", "slideband", "shaft", "air"]
class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}
elem_color_idx = np.array([class_to_idx[c] for c in elem_class])

# triang(triangles_merged 기반) 에 맞는 전체 요소 color index 사용
cmap_reg = matplotlib.colors.ListedColormap([REGION_COLOR[c] for c in CLASS_ORDER])
tc = ax.tripcolor(triang, elem_color_idx, cmap=cmap_reg, vmin=-0.5, vmax=len(CLASS_ORDER)-0.5)
legend_patches = [mpatches.Patch(facecolor=REGION_COLOR[c], label=c) for c in CLASS_ORDER]
ax.legend(handles=legend_patches, loc="upper right", fontsize=7, framealpha=0.8)
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")

ax = axes[1]
ax.set_title("외부 경계 A=0 (Dirichlet)")
ax.tripcolor(triang, a_node, cmap="plasma", shading="gouraud", alpha=0.6)

# triangles_merged 전체 기반으로 edge 카운트
from collections import Counter
edge_count = Counter()
for n1, n2, n3 in triangles_merged.tolist():
    for u, v in [(n1,n2),(n2,n3),(n3,n1)]:
        edge_count[tuple(sorted([u,v]))] += 1
outer_edges = [e for e, cnt in edge_count.items() if cnt == 1]
segs_outer = [[[node_x[u], node_y[u]], [node_x[v], node_y[v]]] for u,v in outer_edges]
lc_outer = LineCollection(segs_outer, colors="red", linewidths=1.5, label="Dirichlet A=0 outer")
ax.add_collection(lc_outer)
ax.legend(fontsize=8, loc="upper right")
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.autoscale_view()

ax = axes[2]
ax.set_title("Interior Edges (학습 그래프 구조)")
ax.set_facecolor("#111111")
inner_edges = [e for e, cnt in edge_count.items() if cnt == 2]
rng = np.random.default_rng(42)
if len(inner_edges) > 50000:
    inner_edges = [inner_edges[i] for i in rng.choice(len(inner_edges), 50000, replace=False)]
segs_inner = [[[node_x[u], node_y[u]], [node_x[v], node_y[v]]] for u,v in inner_edges]
lc_inner = LineCollection(segs_inner, colors="#88aaff", linewidths=0.2, alpha=0.4)
ax.add_collection(lc_inner)
ax.scatter(node_x, node_y, s=0.3, c="white", alpha=0.3)
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.autoscale_view()

plt.tight_layout()
plt.savefig(f"viz_bc_case{CASE_IDX:04d}_step{STEP_IDX}.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"→ 외부 boundary edge 수: {len(outer_edges)}, interior edge 수: {len(inner_edges)}")


In [ ]:
# ── Cell VIZ-3: 주기 경계 조건 (PBC) 시각화 (선 연결 없이 pair 색상으로 표시) ──
import sys
sys.path.insert(0, r"D:\KDH\NvidiaNemo")

from phase1_static.pbc_boundary import extract_periodic_boundary_groups_from_mesh

pos_xy = np.stack([node_x, node_y], axis=1).astype(np.float64)
# PBC 추출: 주 메시 기준 (step-0 connectivity, N_base 절점)
tri_arr = triangles_np.astype(np.int32)
reg_arr = reg_code_elem.astype(np.int32)

group_specs, group_diag, error_code = extract_periodic_boundary_groups_from_mesh(
    pos_xy,
    tri_arr,
    reg_arr,
    moving_reg_codes=moving_reg_codes,
    region_name_by_code=REGION_NAME,
    rotation_deg=-45.0,
)
print(f"PBC group 추출: labels={group_diag.get('group_labels')}, error={error_code}")

sb_mask = np.isin(node_reg, list(SLIDEBAND_CODES))
sb_nodes_x = node_x[sb_mask]
sb_nodes_y = node_y[sb_mask]

origin_xy = np.asarray(group_diag.get("origin_xy", (0.0, 0.0)), dtype=np.float64)
ox, oy = float(origin_xy[0]), float(origin_xy[1])

GROUP_STYLE = {
    "stationary": {"edge": "#27ae60", "label": "고정자 PBC"},
    "moving": {"edge": "#c0392b", "label": "회전자 PBC"},
    "all": {"edge": "#2980b9", "label": "전체 PBC"},
}

# 그룹별 pair를 동일 색으로 표시하기 위한 컬러맵
pair_cmap = plt.get_cmap("tab20")

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(
    f"PBC 시각화(연결선 없음) — Case {CASE_IDX:04d}, step={STEP_IDX}",
    fontsize=13,
    fontweight="bold",
)


def draw_base(ax, alpha_field=0.35):
    ax.tripcolor(triang, a_node, cmap="plasma", shading="gouraud", alpha=alpha_field)
    if "segs_outer" in globals() and len(segs_outer) > 0:
        lc = LineCollection(segs_outer, colors="white", linewidths=0.6, alpha=0.35)
        ax.add_collection(lc)
    ax.set_aspect("equal")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.autoscale_view()


# (a) XY overview: master/slave를 동일 pair 색으로 점만 표시
ax = axes[0]
ax.set_title("XY Overview (● master / ▲ slave, 같은 색=linked pair)")
draw_base(ax)

for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))

    if n_pair > 0:
        colors = [pair_cmap(i % 20) for i in range(n_pair)]
        m_xy = m_xy_all[:n_pair]
        s_xy = s_xy_all[:n_pair]

        ax.scatter(
            m_xy[:, 0],
            m_xy[:, 1],
            c=colors,
            s=18,
            marker="o",
            edgecolors=style["edge"],
            linewidths=0.5,
            alpha=0.95,
            label=f"{style['label']} master",
        )
        ax.scatter(
            s_xy[:, 0],
            s_xy[:, 1],
            c=colors,
            s=20,
            marker="^",
            edgecolors=style["edge"],
            linewidths=0.5,
            alpha=0.95,
            label=f"{style['label']} slave",
        )

    # pair로 못 맞춘 잔여 점은 회색으로 표시
    if len(m_xy_all) > n_pair:
        rem = m_xy_all[n_pair:]
        ax.scatter(rem[:, 0], rem[:, 1], c="lightgray", s=12, marker="o", alpha=0.5)
    if len(s_xy_all) > n_pair:
        rem = s_xy_all[n_pair:]
        ax.scatter(rem[:, 0], rem[:, 1], c="lightgray", s=12, marker="^", alpha=0.5)

if sb_mask.sum() > 0:
    ax.scatter(sb_nodes_x, sb_nodes_y, s=4, c="#9b59b6", alpha=0.25, label="Slide Band")

ax.scatter([ox], [oy], s=70, c="yellow", marker="*", edgecolors="black", linewidths=0.4, zorder=10)
ax.legend(fontsize=7, loc="upper right", framealpha=0.9)


# (b) Polar view: (theta, r)에서 같은 pair 색 표시
ax = axes[1]
ax.set_title("Polar View (theta-r, 같은 색=linked pair)")
for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))
    if n_pair == 0:
        continue

    colors = [pair_cmap(i % 20) for i in range(n_pair)]
    m_xy = m_xy_all[:n_pair]
    s_xy = s_xy_all[:n_pair]

    m_rel = m_xy - origin_xy[None, :]
    s_rel = s_xy - origin_xy[None, :]

    m_th = np.degrees(np.arctan2(m_rel[:, 1], m_rel[:, 0]))
    s_th = np.degrees(np.arctan2(s_rel[:, 1], s_rel[:, 0]))
    m_r = np.linalg.norm(m_rel, axis=1)
    s_r = np.linalg.norm(s_rel, axis=1)

    ax.scatter(m_th, m_r, c=colors, s=18, marker="o", edgecolors=style["edge"], linewidths=0.4, alpha=0.95)
    ax.scatter(s_th, s_r, c=colors, s=20, marker="^", edgecolors=style["edge"], linewidths=0.4, alpha=0.95)

ax.set_xlabel("theta [deg]")
ax.set_ylabel("radius [mm]")
ax.grid(alpha=0.3)


# (c) Slide band zoom: merged_reg + triangles_merged 기반 (per-step SB 포함)
ax = axes[2]
ax.set_title("Slide Band Zoom (점 기반 pair 표현)")
draw_base(ax, alpha_field=0.28)

if sb_mask.sum() > 0:
    # merged_reg 기반으로 SB 요소 선택 (전역 sb_elem_mask 덮어쓰기 방지)
    _sb_in_merged = np.isin(merged_reg, list(SLIDEBAND_CODES))
    if np.any(_sb_in_merged):
        sb_triang_zoom = mtri.Triangulation(node_x, node_y, triangles_merged[_sb_in_merged])
        ax.triplot(sb_triang_zoom, color="#9b59b6", linewidth=0.45, alpha=0.7)
    ax.scatter(sb_nodes_x, sb_nodes_y, s=6, c="#9b59b6", zorder=4, alpha=0.5)

for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))
    if n_pair == 0:
        continue

    colors = [pair_cmap(i % 20) for i in range(n_pair)]
    m_xy = m_xy_all[:n_pair]
    s_xy = s_xy_all[:n_pair]

    ax.scatter(m_xy[:, 0], m_xy[:, 1], c=colors, s=24, marker="o", edgecolors=style["edge"], linewidths=0.5, zorder=6)
    ax.scatter(s_xy[:, 0], s_xy[:, 1], c=colors, s=26, marker="^", edgecolors=style["edge"], linewidths=0.5, zorder=6)

if sb_mask.sum() > 0:
    xc = float(np.mean(sb_nodes_x))
    yc = float(np.mean(sb_nodes_y))
    r_sb = float(max(np.abs(sb_nodes_x - xc).max(), np.abs(sb_nodes_y - yc).max()) * 1.5 + 2.0)
    ax.set_xlim(xc - r_sb, xc + r_sb)
    ax.set_ylim(yc - r_sb, yc + r_sb)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=6, label="master"),
    plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=6, label="slave"),
    mpatches.Patch(facecolor="#9b59b6", label="Slide Band"),
]
ax.legend(handles=legend_handles, fontsize=7, loc="upper right", framealpha=0.9)

plt.tight_layout()
out_path = f"viz_pbc_case{CASE_IDX:04d}_step{STEP_IDX}_paircolor.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"→ 저장: {out_path}")
print(f"그룹 라벨: {group_diag.get('group_labels')}, 그룹 수: {len(group_specs)}")


# 학습 수행

## 11) Full 40-case DOE 학습

`train_manifest.json` 의 전체 40개 케이스를 사용해 SymMGN을 본격 학습합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | **40개** (train_manifest.json 전체) |
| Source file type | OnLoadTorque |
| Max steps/case | 1 (static 첫 step) |
| Epochs | 50 |
| Hidden dim | 128 |
| Batch size | 4 |

> ⚠ GPU 시간이 상당히 소요됩니다. 진행 상황은 로그 파일에서 확인하세요.


In [ ]:
# ── TensorBoard 런치 셀 ─────────────────────────────────────────────────────
# 학습 셀 실행 전 이 셀을 먼저 실행하면 브라우저에서 실시간 loss 확인 가능
# 학습 중에도 실행 가능 (자동 갱신)
import subprocess, pathlib, webbrowser, time

ROOT = globals().get("ROOT", pathlib.Path.cwd())
TB_LOG_DIR_HOST = ROOT / "logs" / "tb_full40"   # 호스트 경로 (마운트됨)
TB_LOG_DIR_HOST.mkdir(parents=True, exist_ok=True)

TB_PORT = 6006

# 이미 실행 중인 TensorBoard가 있으면 재사용
import socket
def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

if _port_in_use(TB_PORT):
    print(f"TensorBoard 이미 실행 중 → http://localhost:{TB_PORT}")
else:
    # 호스트에서 직접 실행 (PyMotorEnv_310에 tensorboard 포함)
    tb_proc = subprocess.Popen(
        [
            r"C:\Users\moa\.ansys_python_venvs\PyMotorEnv_310\Scripts\tensorboard.exe",
            "--logdir", str(TB_LOG_DIR_HOST),
            "--port", str(TB_PORT),
            "--reload_interval", "10",   # 10초마다 갱신
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(3)
    if _port_in_use(TB_PORT):
        print(f"TensorBoard 시작됨 → http://localhost:{TB_PORT}")
    else:
        print("⚠ TensorBoard 시작 실패. 경로 확인:")
        print(f"  logdir = {TB_LOG_DIR_HOST}")

print(f"TB log dir: {TB_LOG_DIR_HOST}")
print("학습 셀의 --tb-log-dir 가 같은 경로를 가리키면 실시간 갱신됩니다.")


### 11-A) 속도 진단 (5 케이스 × 3 스텝 × 5 epoch)

Full 학습 전 1 epoch 소요 시간을 측정합니다.
결과에 따라 batch_size / hidden_dim / max_steps_per_case 조정 여부를 결정합니다.


In [ ]:
# ── 속도 진단: 5케이스 × 3스텝 × 5 epoch ──────────────────────────────────
# 목표: epoch당 소요 시간 측정 → Full 학습 예상 시간 산출
import time as _time

DIAG_CASE_INDICES = list(range(5))
DIAG_MAX_STEPS    = 10
DIAG_EPOCHS       = 5
DIAG_BATCH_SIZE   = 4
DIAG_HIDDEN_DIM   = 128

diag_cmd = " ".join([
    "python -m phase1_static.train",
    "--input-format doe",
    "--data-dir /workspace/doe_data",
    "--case-indices " + " ".join(str(i) for i in DIAG_CASE_INDICES),
    "--source-file-types OnLoadTorque",
    f"--max-steps-per-case {DIAG_MAX_STEPS}",
    f"--epochs {DIAG_EPOCHS}",
    f"--batch-size {DIAG_BATCH_SIZE}",
    f"--hidden-dim {DIAG_HIDDEN_DIM}",
    "--seed 42",
    "--log-file logs/diag_epoch.jsonl",
])

# 이전 진단 로그 제거
import subprocess
subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "bash", "-c",
     "rm -f /workspace/app/logs/diag_epoch.jsonl"],
    capture_output=True
)

t0 = _time.time()
cp_diag = run_docker(diag_cmd, check=False)
wall_s = _time.time() - t0

# epoch 로그 분석
r = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "bash", "-c",
     "cat /workspace/app/logs/diag_epoch.jsonl 2>/dev/null"],
    capture_output=True, text=True
)
diag_logs = [json.loads(l) for l in r.stdout.splitlines() if l.strip()]

print(f"\n=== 속도 진단 결과 ===")
print(f"Wall time: {wall_s:.1f}s ({wall_s/60:.1f}min)")
print(f"완료 epoch: {len(diag_logs)}/{DIAG_EPOCHS}")
print(f"returncode: {cp_diag.returncode}")

if diag_logs:
    sec_per_epoch = wall_s / len(diag_logs)
    first = diag_logs[0]
    last  = diag_logs[-1]
    print(f"\nepoch당 시간: {sec_per_epoch:.1f}s ({sec_per_epoch/60:.1f}min)")
    print(f"train_total: {first['train_total']:.6f} → {last['train_total']:.6f}")
    print(f"val_total  : {first['val_total']:.6f} → {last['val_total']:.6f}")

    # Full 학습 예상
    scale = (40 / len(DIAG_CASE_INDICES)) * (10 / DIAG_MAX_STEPS)
    full_epoch_min = sec_per_epoch * scale / 60
    full_100_hr = full_epoch_min * 100 / 60
    print(f"\n[Full 40케이스 × 10스텝 예상]")
    print(f"  epoch당: ~{full_epoch_min:.0f}분")
    print(f"  100 epoch 총: ~{full_100_hr:.1f}시간")
else:
    print("\n[경고] epoch 로그가 비어있습니다. stderr를 확인하세요:")
    print((cp_diag.stderr or "")[-500:])


### 11-B) 속도 진단 모델 추론

5케이스 × 10스텝으로 학습한 진단 모델(5 epoch)로 동일 케이스를 추론합니다.
`results/diag_infer/` 에 NPZ를 저장합니다.

In [ ]:
# ── 속도 진단 모델 추론: 5케이스 × 스텝별 ──────────────────────────────────
import json
import subprocess
import textwrap
import time as _time
from pathlib import Path

import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
CONTAINER_NAME = globals().get("CONTAINER_NAME", "motor_compare")

DIAG_CASE_INDICES = globals().get("DIAG_CASE_INDICES", list(range(5)))
DIAG_CKPT_NAME = globals().get("DIAG_CKPT_NAME", "diag_5case_ckpt.pt")
DIAG_INFER_DIR = ROOT / "results" / "diag_infer"
DIAG_INFER_DIR.mkdir(parents=True, exist_ok=True)
DIAG_INFER_STEPS = [1, 2, 3]  # step slots to infer

DIAG_INFER_SCRIPT = textwrap.dedent(
    """
    import json, os, sys, time
    from pathlib import Path
    import numpy as np, torch

    os.chdir("/workspace/app")
    sys.path.insert(0, "/workspace/app")

    from infer_phase1_pbc import (
        CHANNEL_ORDER, compute_per_channel_metrics,
        load_symm_mgn, run_inference, save_infer_npz,
    )
    from phase1_static.motor_dataset import build_samples_from_doe_manifest

    CASE_INDICES = __CASE_INDICES__
    STEP_SLOTS = __STEP_SLOTS__
    SOURCE_FILE_TYPES = ["OnLoadTorque"]
    CKPT_PATH = Path("/workspace/app/results/__CKPT_NAME__")
    OUT_DIR = Path("/workspace/app/results/diag_infer")

    script_t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    model, ckpt = load_symm_mgn(CKPT_PATH, device)
    channel_order = ckpt.get("channel_order", CHANNEL_ORDER)
    channel_keys = [name.lower() for name in channel_order]
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    summaries = []

    for case_idx in CASE_INDICES:
        samples = build_samples_from_doe_manifest(
            "/workspace/doe_data",
            max_steps_per_case=max(STEP_SLOTS),
            case_indices=[case_idx],
            source_file_types=SOURCE_FILE_TYPES,
        )
        ordered = sorted(samples, key=lambda s: int(s.get("step_index", -1)))

        for slot_idx, sample in zip(STEP_SLOTS, ordered[:len(STEP_SLOTS)]):
            t0 = time.time()
            result = run_inference(model, [sample], device, batch_size=1)
            elapsed = time.time() - t0
            metrics = compute_per_channel_metrics(result)

            out_path = OUT_DIR / f"diag_case{case_idx:04d}_step{slot_idx}.npz"
            meta = {
                "model_name": "SymMGN", "ckpt": str(CKPT_PATH),
                "case_idx": case_idx, "step_slot": slot_idx,
                "step_idx": int(sample.get("step_index", -1)),
                "elapsed_ms": round(elapsed * 1000.0, 1),
                "channel_order": channel_order,
                "source_file_types": SOURCE_FILE_TYPES,
            }
            save_infer_npz(out_path, result, metrics, meta)

            summary = {"case_idx": case_idx, "step_slot": slot_idx,
                       "elapsed_ms": round(elapsed * 1000.0, 1)}
            for ch in channel_keys:
                summary[f"rmse_{ch}"] = round(float(metrics[f"rmse_{ch}"]), 6)
            summaries.append(summary)

    # aggregate
    agg = {}
    for ch in channel_keys:
        vals = [s[f"rmse_{ch}"] for s in summaries if f"rmse_{ch}" in s]
        if vals:
            a = np.asarray(vals)
            agg[ch] = {"mean": round(float(a.mean()), 6),
                       "std": round(float(a.std()), 6),
                       "max": round(float(a.max()), 6)}

    peak_gpu_mb = None
    if torch.cuda.is_available():
        peak_gpu_mb = round(torch.cuda.max_memory_allocated(device) / 1024**2, 1)

    print(json.dumps({
        "saved_count": len(summaries),
        "wall_time_s": round(time.time() - script_t0, 3),
        "avg_ms": round(float(np.mean([s["elapsed_ms"] for s in summaries])), 1) if summaries else None,
        "peak_gpu_mb": peak_gpu_mb,
        "aggregate_rmse": agg,
    }, ensure_ascii=False))
    """
).replace("__CASE_INDICES__", json.dumps(DIAG_CASE_INDICES)).replace(
    "__STEP_SLOTS__", json.dumps(DIAG_INFER_STEPS)
).replace(
    "__CKPT_NAME__", DIAG_CKPT_NAME
)

t0 = _time.time()
cp_infer = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", DIAG_INFER_SCRIPT],
    capture_output=True, text=True,
)
infer_wall = _time.time() - t0

print(f"=== 속도 진단 추론 결과 ===")
print(f"returncode: {cp_infer.returncode}  wall: {infer_wall:.1f}s")

if cp_infer.returncode != 0:
    print("[stderr tail]")
    print("\n".join((cp_infer.stderr or "").splitlines()[-30:]))
else:
    # parse last JSON line
    for line in reversed(cp_infer.stdout.splitlines()):
        try:
            infer_summary = json.loads(line)
            print(json.dumps(infer_summary, indent=2, ensure_ascii=False))
            break
        except json.JSONDecodeError:
            continue

    # NPZ 파일 확인
    npz_files = sorted(DIAG_INFER_DIR.glob("diag_case*.npz"))
    print(f"\nNPZ saved: {len(npz_files)} files in {DIAG_INFER_DIR}")
    for f in npz_files[:6]:
        print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

### 11-C) 속도 진단 GT vs Pred + Error Map 시각화

5케이스 × 3스텝 추론 결과를 시각화합니다.
- **상단**: GT vs Pred 필드 비교 (Bx, By, A, Je)
- **하단**: |GT - Pred| 절대 오차 맵

In [ ]:
# ── 속도 진단 GT vs Pred + Error Map ──────────────────────────────────────
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
DIAG_INFER_DIR = globals().get("DIAG_INFER_DIR", ROOT / "results" / "diag_infer")
DIAG_CASE_INDICES = globals().get("DIAG_CASE_INDICES", list(range(5)))
DIAG_VIS_STEPS = globals().get("DIAG_INFER_STEPS", [1, 2, 3])

channels = ["bx", "by", "a", "je"]
labels   = ["Bx", "By", "A", "Je"]

# ── 1) GT vs Pred scatter (대표 step=1 으로 전 케이스 비교) ──────────────────
VIS_STEP = 1  # 시각화할 step slot

fig, axes = plt.subplots(
    len(DIAG_CASE_INDICES), len(channels),
    figsize=(5 * len(channels), 4 * len(DIAG_CASE_INDICES)),
    squeeze=False,
)
fig.suptitle(
    f"속도 진단 — GT(color) vs Pred(outline), step {VIS_STEP}",
    fontsize=14, y=1.01,
)

for row, case_idx in enumerate(DIAG_CASE_INDICES):
    npz_path = DIAG_INFER_DIR / f"diag_case{case_idx:04d}_step{VIS_STEP}.npz"
    if not npz_path.exists():
        for col in range(len(channels)):
            axes[row, col].text(0.5, 0.5, "NPZ 없음", ha="center", va="center")
            axes[row, col].set_axis_off()
        continue

    arr = np.load(npz_path, allow_pickle=True)
    meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    for col, (ch, label) in enumerate(zip(channels, labels)):
        gt   = arr[f"gt_{ch}"]
        pred = arr[f"pred_{ch}"]
        vmin = float(min(gt.min(), pred.min()))
        vmax = float(max(gt.max(), pred.max()))

        ax = axes[row, col]
        sc = ax.scatter(pos_x, pos_y, c=gt, cmap="RdBu_r", s=4, vmin=vmin, vmax=vmax)
        ax.scatter(pos_x, pos_y, c=pred, cmap="RdBu_r", s=1, vmin=vmin, vmax=vmax,
                   marker=".", alpha=0.3)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(label, fontsize=11)
        if col == 0:
            ax.set_ylabel(f"case {case_idx}", fontsize=10)
        plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)

plt.tight_layout()
plt.show()

# ── 2) |GT - Pred| Error Map (전 케이스 × step 1) ───────────────────────────
fig2, axes2 = plt.subplots(
    len(DIAG_CASE_INDICES), len(channels),
    figsize=(5 * len(channels), 4 * len(DIAG_CASE_INDICES)),
    squeeze=False,
)
fig2.suptitle(
    f"속도 진단 — |GT - Pred| Error Map, step {VIS_STEP}",
    fontsize=14, y=1.01,
)

rmse_table = []

for row, case_idx in enumerate(DIAG_CASE_INDICES):
    npz_path = DIAG_INFER_DIR / f"diag_case{case_idx:04d}_step{VIS_STEP}.npz"
    if not npz_path.exists():
        for col in range(len(channels)):
            axes2[row, col].text(0.5, 0.5, "NPZ 없음", ha="center", va="center")
            axes2[row, col].set_axis_off()
        continue

    arr = np.load(npz_path, allow_pickle=True)
    metrics = json.loads(arr["metrics"].item()) if "metrics" in arr.files else {}
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    row_rmse = {"case": case_idx}
    for col, (ch, label) in enumerate(zip(channels, labels)):
        gt   = arr[f"gt_{ch}"]
        pred = arr[f"pred_{ch}"]
        error = np.abs(gt - pred)
        vmax_err = float(np.percentile(error, 95)) or 1e-6

        ax = axes2[row, col]
        sc = ax.scatter(pos_x, pos_y, c=error, cmap="hot_r", s=4, vmin=0, vmax=vmax_err)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        rmse_val = metrics.get(f"rmse_{ch}", float(np.sqrt(np.mean(error**2))))
        ax.text(0.02, 0.98, f"RMSE={rmse_val:.4f}",
                transform=ax.transAxes, fontsize=8, va="top",
                bbox=dict(boxstyle="round", fc="white", alpha=0.8))
        if row == 0:
            ax.set_title(f"|Δ{label}|", fontsize=11)
        if col == 0:
            ax.set_ylabel(f"case {case_idx}", fontsize=10)
        plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        row_rmse[ch] = round(rmse_val, 6)

    rmse_table.append(row_rmse)

plt.tight_layout()
plt.show()

# ── 3) RMSE 요약 테이블 ────────────────────────────────────────────────────
print("\n=== 속도 진단 per-case RMSE (step {}) ===".format(VIS_STEP))
print(f"{'case':>6s}  {'Bx':>10s}  {'By':>10s}  {'A':>10s}  {'Je':>10s}")
for r in rmse_table:
    print(f"{r['case']:6d}  {r.get('bx',0):10.6f}  {r.get('by',0):10.6f}  "
          f"{r.get('a',0):10.6f}  {r.get('je',0):10.6f}")

if rmse_table:
    print(f"\n{'mean':>6s}", end="")
    for ch in channels:
        vals = [r[ch] for r in rmse_table if ch in r]
        print(f"  {np.mean(vals):10.6f}", end="")
    print()

# ── 4) 선택: step 별 추이 (3스텝 모두) ──────────────────────────────────────
if len(DIAG_VIS_STEPS) > 1:
    fig3, axes3 = plt.subplots(1, len(channels), figsize=(5 * len(channels), 4))
    fig3.suptitle("Step별 mean RMSE 추이 (5-case diag)", fontsize=13)

    step_rmses = {ch: [] for ch in channels}
    for step in DIAG_VIS_STEPS:
        for ch in channels:
            vals = []
            for case_idx in DIAG_CASE_INDICES:
                npz_path = DIAG_INFER_DIR / f"diag_case{case_idx:04d}_step{step}.npz"
                if npz_path.exists():
                    arr = np.load(npz_path, allow_pickle=True)
                    m = json.loads(arr["metrics"].item()) if "metrics" in arr.files else {}
                    vals.append(m.get(f"rmse_{ch}", 0))
            step_rmses[ch].append(np.mean(vals) if vals else 0)

    for col, (ch, label) in enumerate(zip(channels, labels)):
        ax = axes3[col] if len(channels) > 1 else axes3
        ax.plot(DIAG_VIS_STEPS, step_rmses[ch], "o-", linewidth=2)
        ax.set_xlabel("step slot")
        ax.set_ylabel("mean RMSE")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import shutil

ROOT = globals().get("ROOT", Path.cwd())
LOG_DIR = globals().get("LOG_DIR", ROOT / "logs" / "tutorial")
LOG_DIR.mkdir(parents=True, exist_ok=True)

FULL_CASE_INDICES = list(range(40))
FULL_SOURCE_FILE_TYPES = ["OnLoadTorque"]
FULL_MAX_STEPS_PER_CASE = 10
FULL_EPOCHS = 100
FULL_BATCH_SIZE = 4
FULL_HIDDEN_DIM = 128
FULL_SEED = 42

FULL_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt"
FULL_TRAIN_LOG = "20_full40_train.log"
FULL_TB_LOG_DIR = ROOT / "logs" / "tb_full40"   # TensorBoard log dir (컨테이너 내 경로와 동일)
FULL_TB_LOG_DIR.mkdir(parents=True, exist_ok=True)

# ckpt가 이미 있으면 기본적으로 재학습 생략
SKIP_IF_CKPT_EXISTS = False

# 재학습할 경우, 기존 ckpt 백업
if (not SKIP_IF_CKPT_EXISTS) and FULL_CKPT_PATH.exists():
    ckpt_backup_dir = ROOT / "results" / "backups" / "ckpt"
    ckpt_backup_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = ckpt_backup_dir / f"{FULL_CKPT_PATH.stem}_{ts}{FULL_CKPT_PATH.suffix}"
    shutil.copy2(FULL_CKPT_PATH, dst)
    print(f"[backup] ckpt -> {dst}")

if SKIP_IF_CKPT_EXISTS and FULL_CKPT_PATH.exists():
    print(f"[skip] 기존 체크포인트 사용: {FULL_CKPT_PATH}")
    print("재학습이 필요하면 SKIP_IF_CKPT_EXISTS = False 로 바꿔서 다시 실행하세요.")
else:
    # 학습 재실행 전 기존 full40 infer npz 백업
    if "backup_npz_dir" in globals():
        backup_npz_dir(ROOT / "results" / "full40_infer", label="before_full40_retrain")

    train_cmd = " ".join(
        [
            "python -m phase1_static.train",
            "--input-format doe",
            "--data-dir /workspace/doe_data",
            "--case-indices " + " ".join(str(i) for i in FULL_CASE_INDICES),
            "--source-file-types " + " ".join(FULL_SOURCE_FILE_TYPES),
            f"--max-steps-per-case {FULL_MAX_STEPS_PER_CASE}",
            f"--epochs {FULL_EPOCHS}",
            f"--batch-size {FULL_BATCH_SIZE}",
            f"--hidden-dim {FULL_HIDDEN_DIM}",
            f"--seed {FULL_SEED}",
            f"--ckpt-out results/{FULL_CKPT_PATH.name}",
            f"--ckpt-interval 10",            # 10 epoch마다 중간 체크포인트 저장
            f"--log-file logs/train_epoch.jsonl",  # epoch별 JSON 로그 (실시간 모니터링)
            f"--tb-log-dir logs/tb_full40",   # TensorBoard: 컨테이너 내 상대 경로
        ]
    )

    cp_full_train = run_docker(train_cmd, check=False)
    save_log(FULL_TRAIN_LOG, cp_full_train)

    combined = (cp_full_train.stdout or "") + "\n" + (cp_full_train.stderr or "")
    epoch_lines = [line for line in combined.splitlines() if "epoch=" in line]

    print(
        json.dumps(
            {
                "returncode": cp_full_train.returncode,
                "case_count": len(FULL_CASE_INDICES),
                "source_file_types": FULL_SOURCE_FILE_TYPES,
                "max_steps_per_case": FULL_MAX_STEPS_PER_CASE,
                "epochs": FULL_EPOCHS,
                "last_epoch": epoch_lines[-1] if epoch_lines else "",
                "ckpt_saved": FULL_CKPT_PATH.exists(),
                "ckpt_size_kb": round(FULL_CKPT_PATH.stat().st_size / 1024, 1) if FULL_CKPT_PATH.exists() else 0,
                "log_file": str(LOG_DIR / FULL_TRAIN_LOG),
                "tb_log_dir": str(FULL_TB_LOG_DIR),
            },
            indent=2,
            ensure_ascii=False,
        )
    )

    if cp_full_train.returncode != 0:
        print("=== train stderr tail ===")
        print("\n".join(combined.splitlines()[-40:]))
        raise RuntimeError(f"Full 40-case 학습 실패. {FULL_TRAIN_LOG} 를 확인하세요.")



## 12) Full 40-case 추론

학습된 Full 40-case 모델로 전체 케이스를 순회하며 추론합니다.
각 case별 per-channel RMSE를 수집해 통계를 생성합니다.

In [ ]:
import json
import subprocess
import textwrap
from pathlib import Path

import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
CONTAINER_NAME = globals().get("CONTAINER_NAME", "motor_compare")
LOG_DIR = globals().get("LOG_DIR", ROOT / "logs" / "tutorial")
LOG_DIR.mkdir(parents=True, exist_ok=True)

FULL_CASE_INDICES = globals().get("FULL_CASE_INDICES", list(range(40)))
FULL_SOURCE_FILE_TYPES = globals().get("FULL_SOURCE_FILE_TYPES", ["OnLoadTorque"])
FULL_CKPT_PATH = globals().get(
    "FULL_CKPT_PATH",
    ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt",
)
FULL_INFER_DIR = ROOT / "results" / "full40_infer"
FULL_INFER_DIR.mkdir(parents=True, exist_ok=True)
FULL_INFER_LOG = "12_full40_infer.log"
FULL_INFER_STEPS = [1, 2, 3]

FULL_INFER_SCRIPT = textwrap.dedent(
    """
    import json
    import os
    import sys
    import time
    from pathlib import Path

    import numpy as np
    import torch

    os.chdir("/workspace/app")
    sys.path.insert(0, "/workspace/app")

    from infer_phase1_pbc import (
        CHANNEL_ORDER,
        compute_per_channel_metrics,
        load_symm_mgn,
        run_inference,
        save_infer_npz,
    )
    from phase1_static.motor_dataset import build_samples_from_doe_manifest

    CASE_INDICES = __CASE_INDICES__
    STEP_SLOTS = __STEP_SLOTS__
    SOURCE_FILE_TYPES = __SOURCE_FILE_TYPES__
    CKPT_PATH = Path("/workspace/app/results/__CKPT_NAME__")
    OUT_DIR = Path("/workspace/app/results/full40_infer")

    script_t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    model, ckpt = load_symm_mgn(CKPT_PATH, device)
    channel_order = ckpt.get("channel_order", CHANNEL_ORDER)
    channel_keys = [name.lower() for name in channel_order]
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    summaries = []
    failed = []

    for loop_idx, case_idx in enumerate(CASE_INDICES, start=1):
        samples = build_samples_from_doe_manifest(
            "/workspace/doe_data",
            max_steps_per_case=max(STEP_SLOTS),
            case_indices=[case_idx],
            source_file_types=SOURCE_FILE_TYPES,
        )
        ordered_samples = sorted(
            samples,
            key=lambda sample: int(sample.get("step_index", -1)),
        )

        for slot_idx, actual_sample in zip(STEP_SLOTS, ordered_samples[: len(STEP_SLOTS)]):
            actual_step_idx = int(actual_sample.get("step_index", -1))

            t0 = time.time()
            result = run_inference(model, [actual_sample], device, batch_size=1)
            elapsed = time.time() - t0
            metrics = compute_per_channel_metrics(result)

            out_path = OUT_DIR / f"full40_case{case_idx:04d}_step{slot_idx}.npz"
            meta = {
                "model_name": "SymMGN",
                "ckpt": str(CKPT_PATH),
                "case_idx": case_idx,
                "step_slot": slot_idx,
                "step_idx": actual_step_idx,
                "n_samples": 1,
                "elapsed_s": round(elapsed, 3),
                "elapsed_ms": round(elapsed * 1000.0, 1),
                "channel_order": channel_order,
                "pbc_enabled": True,
                "pbc_rotation_deg": -45.0,
                "source_file_types": SOURCE_FILE_TYPES,
                "source_file_name": actual_sample.get("source_file_name", ""),
                "step_semantics": actual_sample.get("step_semantics", ""),
                "time_s": float(actual_sample.get("time_s", 0.0)),
                "rotate_step": float(actual_sample.get("rotate_step", 0.0)),
            }
            save_infer_npz(out_path, result, metrics, meta)

            summary = {
                "case_idx": case_idx,
                "step_slot": slot_idx,
                "step_idx": actual_step_idx,
                "elapsed_ms": round(elapsed * 1000.0, 1),
                "size_kb": round(out_path.stat().st_size / 1024.0, 1),
            }
            for channel_name in channel_keys:
                summary[f"rmse_{channel_name}"] = round(
                    float(metrics[f"rmse_{channel_name}"]),
                    6,
                )
            summaries.append(summary)

        if len(ordered_samples) < len(STEP_SLOTS):
            for slot_idx in STEP_SLOTS[len(ordered_samples):]:
                failed.append(
                    {
                        "case_idx": case_idx,
                        "step_slot": slot_idx,
                        "reason": "missing_sample",
                    }
                )

        if loop_idx % 10 == 0 or loop_idx == len(CASE_INDICES):
            print(json.dumps({"progress": f"{loop_idx}/{len(CASE_INDICES)}"}, ensure_ascii=False))

    aggregate_rmse = {}
    for channel_name in channel_keys:
        values = [
            float(item[f"rmse_{channel_name}"])
            for item in summaries
            if f"rmse_{channel_name}" in item
        ]
        if values:
            values_arr = np.asarray(values, dtype=float)
            aggregate_rmse[channel_name] = {
                "mean": round(float(values_arr.mean()), 6),
                "std": round(float(values_arr.std()), 6),
                "max": round(float(values_arr.max()), 6),
            }

    peak_gpu_mb = None
    if torch.cuda.is_available():
        peak_gpu_mb = round(torch.cuda.max_memory_allocated(device) / (1024.0 ** 2), 1)

    avg_elapsed_ms = None
    if summaries:
        avg_elapsed_ms = round(
            float(np.mean([item["elapsed_ms"] for item in summaries])),
            1,
        )

    print(
        json.dumps(
            {
                "requested_cases": len(CASE_INDICES),
                "requested_step_slots": STEP_SLOTS,
                "saved_count": len(summaries),
                "failed": failed,
                "wall_time_s": round(time.time() - script_t0, 3),
                "avg_elapsed_ms": avg_elapsed_ms,
                "peak_gpu_mb": peak_gpu_mb,
                "aggregate_rmse": aggregate_rmse,
            },
            ensure_ascii=False,
        )
    )
    """
).replace("__CASE_INDICES__", json.dumps(FULL_CASE_INDICES)).replace(
    "__STEP_SLOTS__", json.dumps(FULL_INFER_STEPS)
).replace(
    "__SOURCE_FILE_TYPES__", json.dumps(FULL_SOURCE_FILE_TYPES)
).replace(
    "__CKPT_NAME__", FULL_CKPT_PATH.name
)

cp_full_infer = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", FULL_INFER_SCRIPT],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)

save_log_fn = globals().get("save_log")
if callable(save_log_fn):
    save_log_fn(FULL_INFER_LOG, cp_full_infer)
else:
    (LOG_DIR / FULL_INFER_LOG).write_text(
        (cp_full_infer.stdout or "") + "\n" + (cp_full_infer.stderr or ""),
        encoding="utf-8",
    )

stdout_lines = [
    line.strip()
    for line in (cp_full_infer.stdout or "").splitlines()
    if line.strip()
]
parsed_lines = []
for line in stdout_lines:
    try:
        parsed_lines.append(json.loads(line))
    except json.JSONDecodeError:
        print(line)

summary = next((item for item in reversed(parsed_lines) if "saved_count" in item), {})
progress_updates = [item["progress"] for item in parsed_lines if "progress" in item]

print("=== Full 40-case Inference Summary ===")
print(
    json.dumps(
        {
            "returncode": cp_full_infer.returncode,
            "progress_updates": progress_updates,
            **summary,
            "log_file": str(LOG_DIR / FULL_INFER_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

expected_count = len(FULL_CASE_INDICES) * len(FULL_INFER_STEPS)
if cp_full_infer.returncode != 0:
    print("=== infer stderr tail ===")
    print("\n".join((cp_full_infer.stderr or "").splitlines()[-40:]))
    raise RuntimeError("Full 40-case SymMGN 추론 실패. 12_full40_infer.log를 확인하세요.")

if summary.get("saved_count", 0) != expected_count:
    raise RuntimeError(
        f"Step별 NPZ 저장 수가 예상과 다릅니다: "
        f"{summary.get('saved_count', 0)} / {expected_count}"
    )

if summary.get("failed"):
    raise RuntimeError(
        "일부 case/step 저장이 실패했습니다. "
        "12_full40_infer.log를 확인하세요."
    )


## 13) Full 40-case GT vs Pred 시각화

대표 case 4개를 골라 GT vs Pred scatter를 비교합니다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = globals().get("VIS_CASE_INDICES", [0, 1, 2, 3])
VIS_STEP_INDICES = globals().get("FULL_INFER_STEPS", [1, 2, 3])

channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]

for case_idx in VIS_CASE_INDICES:
    for step_idx in VIS_STEP_INDICES:
        npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step{step_idx}.npz"
        if not npz_path.exists():
            print(f"case {case_idx} step {step_idx}: NPZ 없음, skip")
            continue

        arr = np.load(npz_path, allow_pickle=True)
        meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
        actual_step_idx = int(meta.get("step_idx", step_idx))
        pos_x = arr["pos_x"]
        pos_y = arr["pos_y"]

        fig, axes = plt.subplots(2, len(channels), figsize=(22, 7))
        fig.suptitle(
            f"case {case_idx:04d} - step {actual_step_idx} GT (top) vs Pred (bottom)"
            ,fontsize=13,
        )

        for col_idx, (ch, label) in enumerate(zip(channels, labels)):
            gt = arr[f"gt_{ch}"]
            pred = arr[f"pred_{ch}"]
            vmin = float(min(gt.min(), pred.min()))
            vmax = float(max(gt.max(), pred.max()))

            sc_gt = axes[0, col_idx].scatter(
                pos_x,
                pos_y,
                c=gt,
                cmap="RdBu_r",
                s=6,
                vmin=vmin,
                vmax=vmax,
            )
            axes[0, col_idx].set_title(f"GT {label}")
            axes[0, col_idx].set_aspect("equal")
            axes[0, col_idx].set_xticks([])
            axes[0, col_idx].set_yticks([])
            plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.04)

            sc_pred = axes[1, col_idx].scatter(
                pos_x,
                pos_y,
                c=pred,
                cmap="RdBu_r",
                s=6,
                vmin=vmin,
                vmax=vmax,
            )
            axes[1, col_idx].set_title(f"Pred {label}")
            axes[1, col_idx].set_aspect("equal")
            axes[1, col_idx].set_xticks([])
            axes[1, col_idx].set_yticks([])
            plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.04)

        plt.tight_layout()
        save_path = ROOT / "logs" / f"full40_case{case_idx:04d}_step{actual_step_idx}_gt_vs_pred.png"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=110, bbox_inches="tight")
        plt.show()
        plt.close()
        print(
            json.dumps(
                {
                    "case_idx": case_idx,
                    "step_idx": actual_step_idx,
                    "saved_png": str(save_path),
                },
                ensure_ascii=False,
            )
        )

## 13) Full 40-case GT vs Pred 시각화

대표 case 4개를 골라 GT vs Pred scatter를 비교합니다.


## 13-B) Full 40-case |GT - Pred| Error Map

동일 대표 case 4개에 대해 채널별 absolute error를 시각화합니다.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = [0, 10, 20, 30]  # 대표 4개 case
# 학습 채널 4개 (j_raw는 추론 NPZ에 포함되지 않음)
channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]

fig, axes = plt.subplots(
    len(VIS_CASE_INDICES),
    len(channels),
    figsize=(16, 4 * len(VIS_CASE_INDICES)),
)
axes = np.atleast_2d(axes)
fig.suptitle("Full 40-case SymMGN — |GT - Pred| error map", fontsize=14)

for row_idx, case_idx in enumerate(VIS_CASE_INDICES):
    npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step1.npz"
    if not npz_path.exists():
        for col_idx in range(len(channels)):
            axes[row_idx, col_idx].text(
                0.5,
                0.5,
                f"case {case_idx}\nNPZ 없음",
                ha="center",
                va="center",
            )
            axes[row_idx, col_idx].set_axis_off()
        continue

    arr = np.load(npz_path, allow_pickle=True)
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    for col_idx, (ch, label) in enumerate(zip(channels, labels)):
        gt_key   = f"gt_{ch}"
        pred_key = f"pred_{ch}"
        if gt_key not in arr.files or pred_key not in arr.files:
            axes[row_idx, col_idx].text(0.5, 0.5, f"{ch}\n없음", ha="center", va="center")
            axes[row_idx, col_idx].set_axis_off()
            continue

        gt = arr[gt_key]
        pred = arr[pred_key]
        error = np.abs(gt - pred)
        vmax_err = float(np.percentile(error, 95))

        sc = axes[row_idx, col_idx].scatter(
            pos_x,
            pos_y,
            c=error,
            cmap="hot_r",
            s=6,
            vmin=0,
            vmax=vmax_err,
        )
        axes[row_idx, col_idx].set_aspect("equal")
        axes[row_idx, col_idx].set_xticks([])
        axes[row_idx, col_idx].set_yticks([])
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"|GT-Pred| {label}")
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(f"case {case_idx:04d}")
        plt.colorbar(sc, ax=axes[row_idx, col_idx], fraction=0.04)

plt.tight_layout()
vis_path = ROOT / "logs" / "full40_gt_vs_pred_error.png"
vis_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(vis_path, dpi=110, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {vis_path}")


## 13-C) Anti-Periodic 대칭을 이용한 Full Motor 복원

1/8 섹터 추론 결과를 8회 회전 + anti-periodic sign flip으로 full 360° 모터 단면을 복원합니다.

**변환 규칙 (섹터 k = 0…7, α = 45°):**
- 위치: $(x, y)$ → 회전 $k\alpha$
- 스칼라 (A, J): $(-1)^k$ 부호 반전
- 벡터 (Bx, By): $(-1)^k$ × 회전 행렬 적용

$$\begin{bmatrix} B_x' \\ B_y' \end{bmatrix} = (-1)^k \begin{bmatrix} \cos k\alpha & -\sin k\alpha \\ \sin k\alpha & \cos k\alpha \end{bmatrix} \begin{bmatrix} B_x \\ B_y \end{bmatrix}$$

## 14) Phase 1 완료 Evidence Summary

Phase 1 (Static SymMGN with PBC) 완료 체크리스트:

| 항목 | 상태 |
|------|------|
| Contract 경계 테스트 통과 | ✅ 위 셀에서 확인 |
| PBC 경계/계약 호스트 테스트 | ✅ 위 셀에서 확인 |
| Overfit-Single 게이트 통과 | ✅ 위 셀에서 확인 |
| PBC 가시화 (학습 전) | ✅ 위 셀에서 확인 |
| 3-case Smoke Test 통과 | ✅ 위 셀에서 확인 |
| Full 40-case 학습 완료 | ⬜ 위 셀 실행 후 체크 |
| Full 40-case 추론 + RMSE 통계 | ⬜ 위 셀 실행 후 체크 |
| GT vs Pred 시각화 | ⬜ 위 셀 실행 후 체크 |

Phase 1이 완료되면 `phase2_tutorial.ipynb`로 진행합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
FULL_VIS_CASE = 0

SECTOR_COUNT = 8
SECTOR_ANGLE_DEG = 45.0


def reconstruct_full_motor(pos_x, pos_y, fields_dict, n_sectors=8, angle_deg=45.0):
    all_x, all_y = [], []
    all_fields = {key: [] for key in fields_dict}

    for sector_idx in range(n_sectors):
        rad = np.radians(sector_idx * angle_deg)
        cos_k = np.cos(rad)
        sin_k = np.sin(rad)
        sign = (-1.0) ** sector_idx

        rot_x = pos_x * cos_k - pos_y * sin_k
        rot_y = pos_x * sin_k + pos_y * cos_k
        all_x.append(rot_x)
        all_y.append(rot_y)

        bx_orig = fields_dict["bx"]
        by_orig = fields_dict["by"]
        all_fields["bx"].append(sign * (bx_orig * cos_k - by_orig * sin_k))
        all_fields["by"].append(sign * (bx_orig * sin_k + by_orig * cos_k))
        all_fields["a"].append(sign * fields_dict["a"])
        all_fields["je"].append(sign * fields_dict["je"])

    full_x = np.concatenate(all_x)
    full_y = np.concatenate(all_y)
    full_fields = {key: np.concatenate(values) for key, values in all_fields.items()}
    return full_x, full_y, full_fields


npz_path = FULL_INFER_DIR / f"full40_case{FULL_VIS_CASE:04d}_step1.npz"
if not npz_path.exists():
    raise FileNotFoundError(f"NPZ not found: {npz_path}")

arr = np.load(npz_path, allow_pickle=True)
sector_x = arr["pos_x"]
sector_y = arr["pos_y"]

channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]
gt_fields = {channel: arr[f"gt_{channel}"] for channel in channels}
pred_fields = {channel: arr[f"pred_{channel}"] for channel in channels}

gt_full_x, gt_full_y, gt_full = reconstruct_full_motor(sector_x, sector_y, gt_fields)
pred_full_x, pred_full_y, pred_full = reconstruct_full_motor(sector_x, sector_y, pred_fields)

fig, axes = plt.subplots(2, len(channels), figsize=(24, 10))
fig.suptitle(
    f"Full Motor (8 sectors) — case {FULL_VIS_CASE:04d} GT (top) vs Pred (bottom)",
    fontsize=14,
)

for col_idx, (channel, label) in enumerate(zip(channels, labels)):
    gt_vals = gt_full[channel]
    pred_vals = pred_full[channel]
    vmin = float(min(gt_vals.min(), pred_vals.min()))
    vmax = float(max(gt_vals.max(), pred_vals.max()))

    sc_gt = axes[0, col_idx].scatter(
        gt_full_x,
        gt_full_y,
        c=gt_vals,
        cmap="RdBu_r",
        s=1.5,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.046)

    sc_pred = axes[1, col_idx].scatter(
        pred_full_x,
        pred_full_y,
        c=pred_vals,
        cmap="RdBu_r",
        s=1.5,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.046)

plt.tight_layout()
save_path = ROOT / "logs" / f"full_motor_case{FULL_VIS_CASE:04d}_gt_vs_pred.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {save_path}")
print(f"total nodes: {len(gt_full_x)} ({len(sector_x)} x {SECTOR_COUNT} sectors)")